# Facial Emotion Recognition — MediaPipe 52 Blendshape Pipeline
**Refactored from:** 1404-dim Raw Landmark Residual MLP  
**Refactored to:** 52-dim Muscle-Based Blendshape Compact MLP  
**Target:** TFLite INT8 for mobile NPU, real-time inference  
**Datasets:** RAF-DB (in-the-wild) + AffectNet (large-scale)  

---
### Why Blendshapes vs Raw Landmarks?
| Dimension | Raw Landmarks | Blendshapes |
|-----------|--------------|-------------|
| Feature count | 1404 (468 pts × 3) | **52** |
| Semantic meaning | Geometric (XYZ coords) | **Muscle activation coefficients** |
| Pose invariance | Low (head rotation bleeds into coords) | **High** (normalized by MediaPipe) |
| In-the-wild robustness | Requires augmentation | **Built-in** |
| Model size | ~2MB+ | **<500KB** |

Blendshapes like `cheekPuff`, `jawOpen`, `browDownLeft` directly encode
the FACS-aligned muscle movements that define emotion — the exact signal
a CNN extracts from pixels, but pre-computed at zero extra cost.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted.')

Mounted at /content/drive
✅ Google Drive mounted.


In [ ]:
import zipfile, os

zip_path   = "/content/testing rafdb.zip"
extract_to = "/content/"

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_to)
    print("✅ Unzipped successfully.")
else:
    print(f"⚠️  Zip not found at {zip_path}. If already extracted, ignore this.")

✅ Unzipped successfully.


In [ ]:
# import shutil
# import os
# from pathlib import Path

# # Config
# FINETUNE_ROOT = Path("AffectNetCustom")
# SPLITS = ["train", "val", "test"]
# NEGATIVES = ["Anger", "Fear", "Sad"]
# TARGET_CLASS = "Unconfident"

# for split in SPLITS:
#     split_dir = FINETUNE_ROOT / split
#     target_dir = split_dir / TARGET_CLASS
#     target_dir.mkdir(parents=True, exist_ok=True)

#     print(f"📦 Processing {split} split...")
#     for neg_cls in NEGATIVES:
#         src_dir = split_dir / neg_cls
#         if not src_dir.exists():
#             continue

#         files = list(src_dir.glob("*"))
#         for f in files:
#             # Move and rename to avoid collisions
#             shutil.move(str(f), str(target_dir / f"{neg_cls}_{f.name}"))

#         # Remove empty folder
#         src_dir.rmdir()
#         print(f"  ✅ Merged {neg_cls} into {TARGET_CLASS}")

# print("\n🎉 Folders restructured for 3-class system.")

📦 Processing train split...
  ✅ Merged Anger into Unconfident
  ✅ Merged Fear into Unconfident
  ✅ Merged Sad into Unconfident
📦 Processing val split...
  ✅ Merged Anger into Unconfident
  ✅ Merged Fear into Unconfident
  ✅ Merged Sad into Unconfident
📦 Processing test split...
  ✅ Merged Anger into Unconfident
  ✅ Merged Fear into Unconfident
  ✅ Merged Sad into Unconfident

🎉 Folders restructured for 3-class system.


In [ ]:
# import os
# import shutil
# import random
# from pathlib import Path

# # Paths
# datasetA = Path("/content/AffectNetCustom")
# datasetB = Path("/content/testing rafdb")
# output = Path("final_dataset")

# classes = ["Anger", "Fear", "Happy", "Neutral", "Sad"]
# splits = ["train", "val", "test"]

# # Create folders
# for split in splits:
#     for cls in classes:
#         (output / split / cls).mkdir(parents=True, exist_ok=True)

# # ---------- Helper function ----------
# def copy_with_rename(src_files, dest_dir, prefix):
#     for i, file in enumerate(src_files):
#         new_name = f"{prefix}_{i}_{file.name}"
#         shutil.copy(file, dest_dir / new_name)

# # ---------- 1. Copy Dataset A ----------
# for split in splits:
#     for cls in classes:
#         src = datasetA / split / cls
#         files = list(src.glob("*"))
#         dest = output / split / cls

#         copy_with_rename(files, dest, prefix=f"A_{split}_{cls}")

# # ---------- 2. Process Dataset B ----------
# split_ratio = {"train": 0.8, "val": 0.1, "test": 0.1}

# for cls in classes:
#     src = datasetB / cls   # ✅ FIXED PATH

#     if not src.exists():
#         print(f"❌ Missing class folder in Dataset B: {cls}")
#         continue

#     files = list(src.glob("*"))
#     print(f"✅ {cls}: {len(files)} images found")

#     random.shuffle(files)

#     n = len(files)
#     n_train = int(n * split_ratio["train"])
#     n_val = int(n * split_ratio["val"])

#     train_files = files[:n_train]
#     val_files = files[n_train:n_train+n_val]
#     test_files = files[n_train+n_val:]

#     copy_with_rename(train_files, output / "train" / cls, "B_train_" + cls)
#     copy_with_rename(val_files, output / "val" / cls, "B_val_" + cls)
#     copy_with_rename(test_files, output / "test" / cls, "B_test_" + cls)

# print("✅ Dataset merged, split, and renamed successfully!")

❌ Missing class folder in Dataset B: Anger
❌ Missing class folder in Dataset B: Fear
❌ Missing class folder in Dataset B: Happy
❌ Missing class folder in Dataset B: Neutral
❌ Missing class folder in Dataset B: Sad
✅ Dataset merged, split, and renamed successfully!


In [ ]:
# ── BLOCK 0: Environment Setup ────────────────────────────────────────────────
# Run once to install dependencies
!pip install mediapipe>=0.10.9 tensorflow>=2.15 scikit-learn imbalanced-learn tqdm

import os, sys, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path

print(f"TensorFlow : {tf.__version__}")
print(f"Python     : {sys.version.split()[0]}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")

TensorFlow : 2.19.0
Python     : 3.12.13
GPU devices: []


---
## BLOCK 1 — Data Extraction: MediaPipe FaceLandmarker → 52 Blendshapes

**Key architectural shift:** We call `FaceLandmarker` (Task API, NOT the legacy
`FaceMesh` solution). The Task API is the only one that exposes `face_blendshapes`.

**What the 52 blendshapes represent:**
- **Eye region (16):** browDown/Up/InnerUp, eyeBlink, eyeLook*, eyeSquint, eyeWide
- **Nose (2):** noseSneer Left/Right
- **Cheek (3):** cheekPuff, cheekSquint Left/Right
- **Mouth/Jaw (31):** jawOpen, jawLeft/Right, jawForward, mouthSmile/Frown/
  Pucker/Shrug/Press/Stretch/Upper/Lower/Left/Right/Dimple, etc.

These map almost 1:1 to FACS Action Units — the gold standard for emotion coding.

In [ ]:
# ── BLOCK 1: Data Extraction ──────────────────────────────────────────────────
"""
Extracts 52 MediaPipe face blendshape coefficients and 4 geometric ratios
from AffectNet / RAF-DB.

Expected directory structure:
  dataset_root/
    Anger/       image1.jpg, image2.jpg ...
    Fear/
    Happy/
    Neutral/
    Sad/


Output: blendshapes.csv  (N rows × 58 cols: 56 features + label + source_path)
"""

import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from tqdm import tqdm
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from collections import deque # Added for TemporalFeatureExtractor

# ── Configuration ─────────────────────────────────────────────────────────────
DATASET_ROOT   = Path("./testing rafdb")           # ← change to your path
OUTPUT_CSV     = Path("./blendshapes_temporal_165d.csv") # Updated output CSV name
MODEL_ASSET    = Path("face_landmarker.task")   # download from MediaPipe model hub

# Download: https://storage.googleapis.com/mediapipe-models/face_landmarker/
#           face_landmarker/float16/latest/face_landmarker.task

# Download the model asset if it doesn't exist
if not MODEL_ASSET.exists():
    print(f"Downloading {MODEL_ASSET}...")
    !wget -q -O {MODEL_ASSET} https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task
    print(f"✅ {MODEL_ASSET} downloaded.")

EMOTION_LABELS = {"Anger": 0, "Fear": 1, "Happy": 2, "Neutral": 3, "Sad": 4}
EMOTION_NAMES = ["Anger", "Fear", "Happy", "Neutral", "Sad"]
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

# The canonical blendshape names MediaPipe returns (in order, index 0-51)
_BASE_BLENDSHAPE_NAMES = [
    "_neutral",
    "browDownLeft", "browDownRight", "browInnerUp",
    "browOuterUpLeft", "browOuterUpRight",
    "cheekPuff", "cheekSquintLeft", "cheekSquintRight",
    "eyeBlinkLeft", "eyeBlinkRight",
    "eyeLookDownLeft", "eyeLookDownRight",
    "eyeLookInLeft", "eyeLookInRight",
    "eyeLookOutLeft", "eyeLookOutRight",
    "eyeLookUpLeft", "eyeLookUpRight",
    "eyeSquintLeft", "eyeSquintRight",
    "eyeWideLeft", "eyeWideRight",
    "jawForward", "jawLeft", "jawOpen", "jawRight",
    "mouthClose",
    "mouthDimpleLeft", "mouthDimpleRight",
    "mouthFrownLeft", "mouthFrownRight",
    "mouthFunnel",
    "mouthLeft", "mouthLowerDownLeft", "mouthLowerDownRight",
    "mouthPressLeft", "mouthPressRight",
    "mouthPucker", "mouthRight",
    "mouthRollLower", "mouthRollUpper",
    "mouthShrugLower", "mouthShrugUpper",
    "mouthSmileLeft", "mouthSmileRight",
    "mouthStretchLeft", "mouthStretchRight",
    "mouthUpperUpLeft", "mouthUpperUpRight",
    "noseSneerLeft", "noseSneerRight",
]
_RATIO_NAMES = ["ear", "mar", "brow_to_eye_dist", "mouth_pull"]

# Generate expanded blendshape names for current, delta, and acceleration
EXPANDED_BLENDSHAPE_NAMES = []
for name in _BASE_BLENDSHAPE_NAMES:
    EXPANDED_BLENDSHAPE_NAMES.append(f"{name}_current")
    EXPANDED_BLENDSHAPE_NAMES.append(f"{name}_delta")
    EXPANDED_BLENDSHAPE_NAMES.append(f"{name}_accel")

# Add 5 extra temporal features to reach 161 dimensions from the 52 blendshapes
for i in range(5):
    EXPANDED_BLENDSHAPE_NAMES.append(f"extra_temporal_feat_{i}")

# Combine expanded blendshapes and original ratios
BLENDSHAPE_NAMES = EXPANDED_BLENDSHAPE_NAMES + _RATIO_NAMES # 156 + 5 + 4 = 165 total

# ── Temporal Feature Extractor ────────────────────────────────────────────────
class TemporalFeatureExtractor:
    """
    Expands a blendshape vector to include temporal information (velocity, acceleration).
    Maintains a history of `window_size` frames to compute derivatives.

    Expected output for 52 blendshapes with `window_size=5`:
      - 52 'current' features
      - 52 'delta' (velocity) features
      - 52 'accel' (acceleration) features
      - 5  'extra' features (placeholders for custom temporal features if needed)
      Total: 161 features.
    """
    def __init__(self, window_size: int = 5):
        self.window_size = window_size
        self.history = deque(maxlen=window_size)
        self.num_blendshapes = 52 # Based on MediaPipe's 52 blendshapes

    def extract(self, current_blendshapes: np.ndarray) -> np.ndarray:
        # Ensure input is 52 dimensions
        if current_blendshapes.shape[0] != self.num_blendshapes:
            raise ValueError(
                f"Expected {self.num_blendshapes} blendshapes, got {current_blendshapes.shape[0]}"
            )

        self.history.append(current_blendshapes)

        # Initialize derivatives with zeros if not enough history
        current = current_blendshapes
        delta = np.zeros(self.num_blendshapes, dtype=np.float32)
        accel = np.zeros(self.num_blendshapes, dtype=np.float32)

        if len(self.history) >= 2:
            # Delta (velocity) = current - previous
            prev = self.history[-2]
            delta = current - prev

        if len(self.history) >= 3:
            # Acceleration = (current - prev) - (prev - prev_prev) = current - 2*prev + prev_prev
            prev_prev = self.history[-3]
            accel = current - 2 * prev + prev_prev

        # Placeholder for 5 extra temporal features. For now, zeros.
        extra_features = np.zeros(5, dtype=np.float32)

        # Concatenate all features: current (52) + delta (52) + accel (52) + extra (5) = 161
        expanded_feats = np.concatenate([current, delta, accel, extra_features])
        return expanded_feats

def calculate_ratios(landmarks) -> list:
    """Calculates pose-invariant ratios using 3D IOD normalization."""
    def get_dist(p1_idx, p2_idx):
        # Using 3D coordinates (x, y, z) to handle perspective depth
        p1 = np.array([landmarks[p1_idx].x, landmarks[p1_idx].y, landmarks[p1_idx].z])
        p2 = np.array([landmarks[p2_idx].x, landmarks[p2_idx].y, landmarks[p2_idx].z])
        return np.linalg.norm(p1 - p2)

    # 1. IOD (Inter-Ocular Distance) - Inner corners (133 and 362)
    iod = get_dist(133, 362) + 1e-6

    # 2. EAR & MAR (Inherent Ratios)
    ear = ((get_dist(159, 145) / (get_dist(133, 33) + 1e-6)) +
           (get_dist(386, 374) / (get_dist(362, 263) + 1e-6))) / 2.0
    mar = get_dist(13, 14) / (get_dist(61, 291) + 1e-6)

    # 3. STABILIZED Brow-to-Eye (Crucial for the 'face lift' fix)
    brow_dist = ((get_dist(107, 33) + get_dist(336, 362)) / 2.0) / iod

    # 4. STABILIZED Mouth-Corner Pull
    mouth_pull = get_dist(61, 291) / iod

    return [ear, mar, brow_dist, mouth_pull]

def build_landmarker(model_path: str) -> mp_vision.FaceLandmarker:
    """Constructs a reusable FaceLandmarker in IMAGE mode."""
    base_options = mp_python.BaseOptions(model_asset_path=str(model_path))
    options = mp_vision.FaceLandmarkerOptions(
        base_options=base_options,
        output_face_blendshapes=True,       # ← critical flag
        output_facial_transformation_matrixes=False,
        num_faces=1,
        min_face_detection_confidence=0.4,  # lower threshold for in-the-wild
        min_face_presence_confidence=0.4,
        min_tracking_confidence=0.4,
        running_mode=mp.tasks.vision.RunningMode.IMAGE,
    )
    return mp_vision.FaceLandmarker.create_from_options(options)


def extract_blendshapes_from_image(image_path: Path, landmarker) -> np.ndarray | None:
    """
    Returns a (56,) float32 array of blendshape coefficients and geometric ratios,
    or None if no face detected.

    Robustness strategy for in-the-wild (RAF-DB):
      1. Try original image first
      2. If no detection: CLAHE + slight crop retry
      3. If still none: skip (don't hallucinate features)
    """
    bgr = cv2.imread(str(image_path))
    if bgr is None:
        return None

    def _run_detection(bgr_img):
        rgb = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = landmarker.detect(mp_image)
        if result.face_blendshapes and result.face_landmarks:
            # 52 Blendshapes
            scores = [bs.score for bs in result.face_blendshapes[0]]
            # 4 Ratios
            ratios = calculate_ratios(result.face_landmarks[0])
            return np.array(scores + ratios, dtype=np.float32) # Total: 56
        return None

    # Attempt 1: raw image
    feat = _run_detection(bgr)
    if feat is not None:
        return feat

    # Attempt 2: CLAHE for low-contrast / dark images (common in RAF-DB)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    feat = _run_detection(enhanced)
    if feat is not None:
        return feat

    # Attempt 3: center-crop 90% (removes borders/watermarks in dataset images)
    h, w = bgr.shape[:2]
    m = 0.05
    cropped = bgr[int(h*m):int(h*(1-m)), int(w*m):int(w*(1-m))]
    return _run_detection(cropped)


def extract_dataset(dataset_root: Path, output_csv: Path, model_path: Path):
    """
    Walks dataset_root, extracts blendshapes for every image,
    and saves a CSV. Processes images in sorted order for reproducibility.
    """
    landmarker = build_landmarker(model_path)
    records = []
    skipped = 0

    # --- ADD TO BLOCK 1 ---
    # This expands your vector so the model can see "muscle velocity"
    extractor = TemporalFeatureExtractor(window_size=5)

    for emotion_name, label_idx in EMOTION_LABELS.items():
        emotion_dir = dataset_root / emotion_name # Access 'train' subdirectory
        if not emotion_dir.exists():
            print(f"[WARN] Missing directory: {emotion_dir}")
            continue

        image_paths = sorted(
            p for p in emotion_dir.iterdir()
            if p.suffix.lower() in IMG_EXTENSIONS
        )

        for img_path in tqdm(image_paths, desc=f"{emotion_name:>10}", unit="img"):
            feats_56 = extract_blendshapes_from_image(img_path, landmarker)
            if feats_56 is None:
                skipped += 1
                continue

            # CHANGE: Apply the temporal/semantic expansion
            # This turns your 56-dim vector into a 165-dim vector
            expanded_feats = extractor.extract(feats_56[:52]) # 161 dims
            final_vector = np.concatenate([expanded_feats, feats_56[52:]]) # +4 ratios = 165

            records.append(list(final_vector) + [label_idx, str(img_path)])

    landmarker.close()

    # Updated to 165 features + label + source_path
    columns = BLENDSHAPE_NAMES + ["label", "source_path"]
    df = pd.DataFrame(records, columns=columns)
    df.to_csv(output_csv, index=False)

    total = len(records) + skipped
    print(f"\n✅ Extraction complete")
    print(f"   Saved  : {len(records):,} samples → {output_csv}")
    print(f"   Skipped: {skipped:,} / {total:,} ({100*skipped/max(total,1):.1f}%) — no face detected")
    print(f"\n   Class distribution:")
    print(df['label'].value_counts().sort_index().to_string())
    return df


# ── Run extraction (comment out after first run) ────────────────────────
if not OUTPUT_CSV.exists():
    print("Running data extraction...")
    df = extract_dataset(DATASET_ROOT, OUTPUT_CSV, MODEL_ASSET)
else:
    print("Output CSV already exists, loading from file.")
    df = pd.read_csv(OUTPUT_CSV)

# ── Or load cached CSV ─────────────────────────────


Running data extraction...


       Sad: 100%|██████████| 355/355 [00:05<00:00, 61.58img/s]



✅ Extraction complete
   Saved  : 1,619 samples → blendshapes_temporal_165d.csv
   Skipped: 156 / 1,775 (8.8%) — no face detected

   Class distribution:
label
0    295
1    326
2    343
3    328
4    327


In [ ]:
# --- BLOCK FT: Fine-Tune Only (No Full Retraining) ---
# This block assumes Block 1 (feature extraction utilities) has been run,
# and uses your existing pre-trained checkpoint.

import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
tf.get_logger().setLevel("ERROR")
from sklearn.model_selection import train_test_split
from tqdm import tqdm


required_symbols = [
    "BLENDSHAPE_NAMES",
    "EMOTION_LABELS",
    "IMG_EXTENSIONS",
    "TemporalFeatureExtractor",
    "extract_blendshapes_from_image",
    "build_landmarker",
]
missing_symbols = [s for s in required_symbols if s not in globals()]
if missing_symbols:
    raise ValueError(
        "Missing required symbols from Block 1: " + ", ".join(missing_symbols) +
        ". Run Block 1 extraction cell first."
    )

if "NUM_CLASSES" not in globals():
    NUM_CLASSES = len(EMOTION_LABELS)


# -----------------------------
# Paths / Config
# -----------------------------
PRETRAINED_MODEL_PATH = Path("./best_fer_model.keras")
SCALER_PATH = Path("./blendshape_scaler.pkl")
SCALER_METADATA_CANDIDATES = [
    # Path("./blendshape_metadata.json"),
    # Path("./second training/blendshape_metadata.json"),
    # Path("./first training/blendshape_metadata.json"),
    Path("./third training/blendshape_metadata.json")
]
RAFDB_ROOT = Path("./testing rafdb")  # structure: testing rafdb/Anger|Fear|Happy|Neutral|Sad
RAFDB_FEATURE_CSV = Path("./blendshapes_temporal_165d.csv")
LANDMARKER_ASSET = Path("face_landmarker.task")

BATCH_SIZE = 256
FINE_TUNE_EPOCHS = 30
FINE_TUNE_LR = 5e-5


# -----------------------------
# Helpers
# -----------------------------
def load_pretrained_model(model_path: Path):
    if not model_path.exists():
        raise FileNotFoundError(f"Pretrained model not found: {model_path}")

    print(f"Loading pretrained model: {model_path}")
    try:
        model_obj = tf.keras.models.load_model(
            model_path,
            compile=False,
            safe_mode=False,
        )
    except TypeError:
        model_obj = tf.keras.models.load_model(model_path, compile=False)

    print("Loaded pretrained model.")
    return model_obj


def prepare_model_for_finetuning(model_obj, base_lr=5e-5):
    # Freeze early GLU muscle detector block
    for layer in model_obj.layers:
        if "glu1" in layer.name:
            layer.trainable = False
            print(f"Froze layer: {layer.name}")
        else:
            layer.trainable = True

    optimizer_obj = tf.keras.optimizers.AdamW(
        learning_rate=base_lr,
        weight_decay=1e-4,
        clipnorm=1.0,
    )
    return optimizer_obj


def extract_rafdb_dataset(dataset_root: Path, output_csv: Path, model_path: Path):
    """Extract 165-d temporal features from class-folder RAF-DB layout."""
    if not dataset_root.exists():
        raise FileNotFoundError(f"RAF-DB root not found: {dataset_root}")
    if not model_path.exists():
        raise FileNotFoundError(f"MediaPipe task file not found: {model_path}")

    landmarker = build_landmarker(model_path)
    records = []
    skipped = 0
    extractor = TemporalFeatureExtractor(window_size=5)

    for emotion_name, label_idx in EMOTION_LABELS.items():
        class_dir = dataset_root / emotion_name
        if not class_dir.exists():
            print(f"[WARN] Missing directory: {class_dir}")
            continue

        image_paths = sorted(
            p for p in class_dir.iterdir() if p.suffix.lower() in IMG_EXTENSIONS
        )

        for img_path in tqdm(image_paths, desc=f"RAFDB {emotion_name:>10}", unit="img"):
            feats_56 = extract_blendshapes_from_image(img_path, landmarker)
            if feats_56 is None:
                skipped += 1
                continue

            expanded_161 = extractor.extract(feats_56[:52])
            final_165 = np.concatenate([expanded_161, feats_56[52:]])
            records.append(list(final_165) + [label_idx, str(img_path)])

    landmarker.close()

    columns = BLENDSHAPE_NAMES + ["label", "source_path"]
    df_raf = pd.DataFrame(records, columns=columns)
    df_raf.to_csv(output_csv, index=False)

    total = len(records) + skipped
    print(f"RAF-DB extracted: {len(records):,} / {total:,} (skipped: {skipped:,})")
    return df_raf


def make_supervised_dataset(X, y, batch_size=256, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(X), seed=42)
    ds = ds.batch(batch_size)
    ds = ds.map(
        lambda x, yy: (x, tf.one_hot(tf.cast(yy, tf.int32), NUM_CLASSES)),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    return ds.prefetch(tf.data.AUTOTUNE)


# -----------------------------
# Prepare RAF-DB features
# -----------------------------
if RAFDB_FEATURE_CSV.exists():
    print(f"Loading cached RAF-DB features: {RAFDB_FEATURE_CSV}")
    df_rafdb = pd.read_csv(RAFDB_FEATURE_CSV)
else:
    df_rafdb = extract_rafdb_dataset(RAFDB_ROOT, RAFDB_FEATURE_CSV, LANDMARKER_ASSET)

missing_cols = [c for c in BLENDSHAPE_NAMES if c not in df_rafdb.columns]
if missing_cols:
    raise ValueError(
        "RAF-DB feature CSV is missing required 165-d columns. "
        "Delete rafdb_blendshapes_temporal_165d.csv and rerun this cell to re-extract."
    )

X_raf = df_rafdb[BLENDSHAPE_NAMES].values.astype(np.float32)
y_raf = df_rafdb["label"].values.astype(np.int32)

X_raf_train, X_raf_val, y_raf_train, y_raf_val = train_test_split(
    X_raf,
    y_raf,
    test_size=0.2,
    stratify=y_raf,
    random_state=42,
)


# -----------------------------
# Normalization for fine-tuning
# -----------------------------
class FixedScaler:
    def __init__(self, mean_arr, scale_arr):
        self.mean_ = np.asarray(mean_arr, dtype=np.float32)
        self.scale_ = np.asarray(scale_arr, dtype=np.float32)
        self.scale_[self.scale_ == 0.0] = 1.0

    def transform(self, X):
        return (X - self.mean_) / self.scale_

metadata_path = next((p for p in SCALER_METADATA_CANDIDATES if p.exists()), None)
if metadata_path is not None:
    with open(metadata_path, "r") as f:
        metadata = json.load(f)

    mean_arr = metadata.get("mean")
    scale_arr = metadata.get("scale")
    if mean_arr is None or scale_arr is None:
        raise ValueError(
            f"Metadata file does not contain mean/scale arrays: {metadata_path}"
        )
    if len(mean_arr) != len(BLENDSHAPE_NAMES) or len(scale_arr) != len(BLENDSHAPE_NAMES):
        raise ValueError(
            f"Metadata scaler dims do not match 165 features in {metadata_path}."
        )

    scaler = FixedScaler(mean_arr, scale_arr)
    print(f"Loaded scaler stats from metadata: {metadata_path}")
elif SCALER_PATH.exists():
    with open(SCALER_PATH, "rb") as f:
        scaler = pickle.load(f)
    print(f"Loaded scaler: {SCALER_PATH}")
else:
    print("[WARN] No saved scaler/metadata found. Fitting scaler on RAF-DB train split.")
    from sklearn.preprocessing import StandardScaler

    scaler = StandardScaler()
    scaler.fit(X_raf_train)
    with open(SCALER_PATH, "wb") as f:
        pickle.dump(scaler, f)
    print(f"Saved fallback scaler: {SCALER_PATH}")

X_raf_train = scaler.transform(X_raf_train)
X_raf_val = scaler.transform(X_raf_val)

rafdb_train_ds = make_supervised_dataset(X_raf_train, y_raf_train, batch_size=BATCH_SIZE, shuffle=True)
rafdb_val_ds = make_supervised_dataset(X_raf_val, y_raf_val, batch_size=BATCH_SIZE, shuffle=False)

print(f"Prepared RAF-DB loaders: train={len(X_raf_train):,}, val={len(X_raf_val):,}")


# -----------------------------
# Load model + fine-tune setup
# -----------------------------
model = load_pretrained_model(PRETRAINED_MODEL_PATH)
optimizer = prepare_model_for_finetuning(model, base_lr=FINE_TUNE_LR)

if "embedding_layer" not in [layer.name for layer in model.layers]:
    raise ValueError("embedding_layer not found in pretrained model.")
embedding_model = tf.keras.Model(inputs=model.input, outputs=model.get_layer("embedding_layer").output)


# -----------------------------
# Losses and training steps
# -----------------------------
class CategoricalFocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def call(self, y_true, y_pred_logits):
        y_pred = tf.nn.softmax(y_pred_logits)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        loss = -y_true * tf.math.log(y_pred) * tf.pow(1.0 - y_pred, self.gamma)
        return tf.reduce_sum(loss, axis=-1)


class SupervisedContrastiveLoss(tf.keras.losses.Loss):
    def __init__(self, temperature=0.07, name=None):
        super().__init__(name=name)
        self.temperature = temperature

    def call(self, labels, feature_vectors):
        feature_vectors = tf.math.l2_normalize(feature_vectors, axis=1)
        similarity_matrix = tf.matmul(feature_vectors, feature_vectors, transpose_b=True)

        labels = tf.cast(labels, dtype=tf.int32)
        labels_reshaped = tf.expand_dims(labels, -1)
        label_matrix = tf.cast(tf.equal(labels_reshaped, tf.transpose(labels_reshaped)), tf.float32)

        positive_mask = label_matrix - tf.eye(tf.shape(labels)[0], dtype=tf.float32)
        negative_mask = 1.0 - tf.eye(tf.shape(labels)[0], dtype=tf.float32)

        similarity_matrix = similarity_matrix / self.temperature

        all_other_sim_sum = tf.reduce_sum(tf.exp(similarity_matrix) * negative_mask, axis=1)
        pos_sim_values_sum = tf.reduce_sum(tf.exp(similarity_matrix) * positive_mask, axis=1)

        log_term = tf.math.log(pos_sim_values_sum + 1e-9) - tf.math.log(all_other_sim_sum + 1e-9)
        num_pos_pairs_per_anchor = tf.reduce_sum(positive_mask, axis=1)

        valid_anchors_mask = tf.greater(num_pos_pairs_per_anchor, 0)
        num_valid_anchors = tf.reduce_sum(tf.cast(valid_anchors_mask, tf.float32))

        def true_fn():
            return tf.constant(0.0, dtype=tf.float32)

        def false_fn():
            log_term_masked = tf.boolean_mask(log_term, valid_anchors_mask)
            num_pos_pairs_masked = tf.boolean_mask(num_pos_pairs_per_anchor, valid_anchors_mask)
            return -tf.reduce_sum(log_term_masked / num_pos_pairs_masked) / num_valid_anchors

        return tf.cond(tf.equal(num_valid_anchors, 0), true_fn, false_fn)


def compute_loss(y_true_one_hot, y_true_int, logits, embeddings):
    focal = CategoricalFocalLoss(gamma=2.0)(y_true_one_hot, logits)
    supcon = SupervisedContrastiveLoss(temperature=0.07)(y_true_int, embeddings)
    return focal + (0.3 * supcon)


class MetricTracker:
    def __init__(self):
        self.train_loss = tf.keras.metrics.Mean(name="train_loss")
        self.train_accuracy = tf.keras.metrics.CategoricalAccuracy(name="train_accuracy")
        self.val_loss = tf.keras.metrics.Mean(name="val_loss")
        self.val_accuracy = tf.keras.metrics.CategoricalAccuracy(name="val_accuracy")

    def reset_states(self):
        self.train_loss.reset_state()
        self.train_accuracy.reset_state()
        self.val_loss.reset_state()
        self.val_accuracy.reset_state()


metric_tracker = MetricTracker()


@tf.function
def train_step(data):
    x, y_one_hot = data
    y_int = tf.argmax(y_one_hot, axis=-1, output_type=tf.int32)

    with tf.GradientTape() as tape:
        logits = model(x, training=True)
        embeddings = embedding_model(x, training=True)
        loss = compute_loss(y_one_hot, y_int, logits, embeddings)

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    metric_tracker.train_loss.update_state(loss)
    metric_tracker.train_accuracy.update_state(y_one_hot, logits)


@tf.function
def val_step(data):
    x, y_one_hot = data
    y_int = tf.argmax(y_one_hot, axis=-1, output_type=tf.int32)

    logits = model(x, training=False)
    embeddings = embedding_model(x, training=False)
    loss = compute_loss(y_one_hot, y_int, logits, embeddings)

    metric_tracker.val_loss.update_state(loss)
    metric_tracker.val_accuracy.update_state(y_one_hot, logits)


# -----------------------------
# Fine-tune loop
# -----------------------------
print("\nStarting fine-tuning on RAF-DB...")
best_val_accuracy = 0.0

for epoch in range(FINE_TUNE_EPOCHS):
    metric_tracker.reset_states()

    step_pb = tqdm(
        enumerate(rafdb_train_ds),
        total=len(rafdb_train_ds),
        desc=f"Fine-Tune Epoch {epoch + 1}/{FINE_TUNE_EPOCHS}",
    )
    for step, data in step_pb:
        train_step(data)
        if step % 5 == 0:
            step_pb.set_postfix({"loss": f"{metric_tracker.train_loss.result():.4f}"})

    for data in rafdb_val_ds:
        val_step(data)

    train_acc = float(metric_tracker.train_accuracy.result().numpy())
    val_acc = float(metric_tracker.val_accuracy.result().numpy())
    print(f"Epoch {epoch + 1}: Train Acc = {train_acc:.4f}, Val Acc = {val_acc:.4f}")

    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        model.save("finetuned_fer_model.keras")
        print("Saved best fine-tuned checkpoint: finetuned_fer_model.keras")

print(f"Fine-tuning complete. Best Val Acc: {best_val_accuracy:.4f}")

Loading cached RAF-DB features: blendshapes_temporal_165d.csv
Loaded scaler: blendshape_scaler.pkl
Prepared RAF-DB loaders: train=1,295, val=324
Loading pretrained model: best_fer_model.keras
Loaded pretrained model.
Froze layer: glu1_gate
Froze layer: glu1_linear
Froze layer: glu1_ln

Starting fine-tuning on RAF-DB...


Fine-Tune Epoch 1/30: 100%|██████████| 6/6 [00:39<00:00,  6.65s/it, loss=0.3921]


Epoch 1: Train Acc = 0.6973, Val Acc = 0.7099
Saved best fine-tuned checkpoint: finetuned_fer_model.keras


Fine-Tune Epoch 2/30: 100%|██████████| 6/6 [00:06<00:00,  1.12s/it, loss=0.4602]


Epoch 2: Train Acc = 0.7066, Val Acc = 0.7253
Saved best fine-tuned checkpoint: finetuned_fer_model.keras


...

Epoch 28: Train Acc = 0.7907, Val Acc = 0.7593


Fine-Tune Epoch 29/30: 100%|██████████| 6/6 [00:11<00:00,  1.84s/it, loss=0.2642]


Epoch 29: Train Acc = 0.7907, Val Acc = 0.7685


Fine-Tune Epoch 30/30: 100%|██████████| 6/6 [00:06<00:00,  1.04s/it, loss=0.3122]


Epoch 30: Train Acc = 0.7931, Val Acc = 0.7716
Fine-tuning complete. Best Val Acc: 0.7778


---
## BLOCK 2 — Optimized MLP Architecture

### Design Rationale
```
Input (52) → FC(256) → BN → Swish → Drop(0.3)
          ↓                               ↓
          └──────── residual ─────────────┘  (projection if dims differ)
          → FC(256) → BN → Swish → Drop(0.3)
          ↓                               ↓
          └──────── residual ─────────────┘
          → FC(128) → BN → Swish → Drop(0.2)
          → Output(7, softmax)
```
- **Swish** (x·σ(x)) outperforms ReLU on small tabular-style feature vectors
- **BatchNorm before activation** (not after) preserves blendshape scale info
- **Residual connections** prevent gradient vanishing on narrow bottleneck
- **No bias in FC before BN** — BN's beta parameter absorbs it (saves params)

In [ ]:
# --- BLOCK 2: Model Architecture ---
from tensorflow import keras
from tensorflow.keras import layers, regularizers

INPUT_DIM = 165 # Updated to 165 because of the temporal/semantic expansion
NUM_CLASSES = 5
L2_REG = 1e-4

def glu_block(x, units: int, dropout_rate: float, name: str | None = None):
    # Gated Linear Unit for non-linearity
    gate = layers.Dense(units, activation="sigmoid", name=f"{name}_gate" if name else None)(x)
    linear = layers.Dense(units, name=f"{name}_linear" if name else None)(x)
    h = layers.Multiply()([gate, linear])
    h = layers.LayerNormalization(name=f"{name}_ln" if name else None)(h)
    h = layers.Dropout(dropout_rate)(h)
    return h

def semantic_attention_bottleneck(x, units: int = 224, num_heads: int = 4, name: str | None = None):
    # Simple Multi-Head Self-Attention for semantic tokenization
    # Takes the original 56 blendshapes and outputs a fixed-size embedding

    # Project to attention dimension
    q = layers.Dense(units, name=f"{name}_q" if name else None)(x)
    k = layers.Dense(units, name=f"{name}_k" if name else None)(x)
    v = layers.Dense(units, name=f"{name}_v" if name else None)(x)

    # Multi-head attention (simplified, not full Transformer block)
    attention_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=units // num_heads,
                                                name=f"{name}_mha" if name else None)(q, k, v)

    # Global average pooling to condense into a single vector (224-dim output)
    return layers.GlobalAveragePooling1D(name=f"{name}_gap" if name else None)(attention_output)


def residual_block(x, units: int, dropout_rate: float, name: str):
    h = layers.BatchNormalization(name=f"{name}_bn1")(x)
    h = layers.Activation("swish", name=f"{name}_swish1")(h)
    h = layers.Dense(units, use_bias=False, kernel_initializer="he_uniform",
                     kernel_regularizer=regularizers.l2(L2_REG))(h)
    h = layers.BatchNormalization(name=f"{name}_bn2")(h)
    h = layers.Activation("swish", name=f"{name}_swish2")(h)
    h = layers.Dropout(dropout_rate)(h)
    h = layers.Dense(units, use_bias=False, kernel_initializer="he_uniform",
                     kernel_regularizer=regularizers.l2(L2_REG))(h)

    if x.shape[-1] != units:
        x = layers.Dense(units, use_bias=False, kernel_initializer="he_uniform")(x)
    return layers.Add()([h, x])

def build_fer_model():
    # INPUT_DIM is now 165 because of the expansion above
    inp = layers.Input(shape=(165,), name="features_165d")

    # Stage 1: Noise Filtering via GLU
    x = layers.Dense(128)(inp)
    x = layers.LayerNormalization()(x)
    x = glu_block(x, 128, dropout_rate=0.25, name="glu1")

    # Stage 2: Attention (Groups features into semantic tokens)
    # We take the original 56 "core" features for the attention path
    core_features = layers.Lambda(lambda x: x[:, :56], name="core_features_extract")(inp)
    # Reshape for attention (batch, sequence_length=56, feature_dim=1) -> (batch, 56, 1)
    core_features_reshaped = layers.Reshape((56, 1), name="core_features_reshape")(core_features)
    attn_path = semantic_attention_bottleneck(core_features_reshaped, name="semantic_attn") # 224-dim

    # Stage 3: Fusion
    fused = layers.Concatenate(name="fused_features")([x, attn_path])
    fused = layers.Dense(128, activation='swish', name="embedding_layer")(fused)

    # Stage 4: Classification Head (Outputting Logits for Temperature Scaling)
    fused = glu_block(fused, 64, dropout_rate=0.2, name="glu2")
    out = layers.Dense(NUM_CLASSES, activation=None, name="predictions")(fused)

    return keras.Model(inputs=inp, outputs=out)

# Instantiate the model to update model.summary() output
model = build_fer_model()
model.summary()


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ features_165d       │ (None, 165)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     21,248 │ features_165d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 128)       │        256 │ dense[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ core_features_extr… │ (None, 56)        │          0 │ features_165d[0]… │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ glu1_gate (Dense)   │ (None, 128)       │     16,512 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ glu1_linear (Dense) │ (None, 128)       │     16,512 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ core_features_resh… │ (None, 56, 1)     │          0 │ core_features_ex… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 128)       │          0 │ glu1_gate[0][0],  │
│                     │                   │            │ glu1_linear[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ semantic_attn_q     │ (None, 56, 224)   │        448 │ core_features_re… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ semantic_attn_k     │ (None, 56, 224)   │        448 │ core_features_re… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ semantic_attn_v     │ (None, 56, 224)   │        448 │ core_features_re… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ glu1_ln             │ (None, 128)       │        256 │ multiply[0][0]    │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ semantic_attn_mha   │ (None, 56, 224)   │    201,600 │ semantic_attn_q[… │
│ (MultiHeadAttentio… │                   │            │ semantic_attn_k[… │
│                     │                   │            │ semantic_attn_v[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 128)       │          0 │ glu1_ln[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ semantic_attn_gap   │ (None, 224)       │          0 │ semantic_attn_mh… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fused_features      │ (None, 352)       │          0 │ dropout_4[0][0],  │
│ (Concatenate)       │                   │            │ semantic_attn_ga… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_layer     │ (None, 128)       │     45,184 │ fused_features[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ glu2_gate (Dense)   │ (None, 64)        │      8,256 │ embedding_layer[

 Total params: 319,877 (1.22 MB)

 Trainable params: 319,877 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

---
## BLOCK 3 — Training with Class Imbalance Strategy

### AffectNet Class Imbalance Problem
AffectNet has severe imbalance:
```
Happy:   ~134k  |  Neutral: ~74k  |  Sad: ~25k
Fear:    ~14k   |  Disgust: ~3.8k |  Contempt: ~3.7k  (8-class)
```
Strategy: **Hybrid** — class weights + oversampling (not just one):
1. **`compute_class_weight('balanced')`** for the loss function
2. **`RandomOverSampler`** on minority classes during data prep
3. **Label smoothing (0.1)** — prevents the model from being overconfident
   on the dominant `happy` class
4. **MixUp augmentation** — interpolates blendshape vectors between classes,
   especially effective for the `fear ↔ surprise` confusion boundary

In [ ]:
# ── BLOCK 3: Data Preparation + Training ──────────────────────────────────────
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import RandomOverSampler
import pickle
import tensorflow as tf # Ensure tf is imported for custom training loop
from tensorflow import keras # For AdamW optimizer
from tensorflow.keras import layers, regularizers # Added for model architecture
from pathlib import Path # For OUTPUT_CSV

# --- Re-define global constants used in this block (originally from other cells) ---
# From N8GRF7MQ0mws
OUTPUT_CSV = Path("./blendshapes_temporal_165d.csv")

# Placeholder for blendshape names to ensure BLENDSHAPE_NAMES is defined.
# In a real notebook, these would come from an earlier cell that defines them fully.
# For a standalone fix, we use the known final dimension and structure.
_BASE_BLENDSHAPE_NAMES = [
    "_neutral", "browDownLeft", "browDownRight", "browInnerUp",
    "browOuterUpLeft", "browOuterUpRight", "cheekPuff", "cheekSquintLeft",
    "cheekSquintRight", "eyeBlinkLeft", "eyeBlinkRight", "eyeLookDownLeft",
    "eyeLookDownRight", "eyeLookInLeft", "eyeLookInRight", "eyeLookOutLeft",
    "eyeLookOutRight", "eyeLookUpLeft", "eyeLookUpRight", "eyeSquintLeft",
    "eyeSquintRight", "eyeWideLeft", "eyeWideRight", "jawForward", "jawLeft",
    "jawOpen", "jawRight", "mouthClose", "mouthDimpleLeft", "mouthDimpleRight",
    "mouthFrownLeft", "mouthFrownRight", "mouthFunnel", "mouthLeft",
    "mouthLowerDownLeft", "mouthLowerDownRight", "mouthPressLeft",
    "mouthPressRight", "mouthPucker", "mouthRight", "mouthRollLower",
    "mouthRollUpper", "mouthShrugLower", "mouthShrugUpper", "mouthSmileLeft",
    "mouthSmileRight", "mouthStretchLeft", "mouthStretchRight",
    "mouthUpperUpLeft", "mouthUpperUpRight", "noseSneerLeft", "noseSneerRight",
]
_RATIO_NAMES = ["ear", "mar", "brow_to_eye_dist", "mouth_pull"]

EXPANDED_BLENDSHAPE_NAMES = []
for name in _BASE_BLENDSHAPE_NAMES:
    EXPANDED_BLENDSHAPE_NAMES.append(f"{name}_current")
    EXPANDED_BLENDSHAPE_NAMES.append(f"{name}_delta")
    EXPANDED_BLENDSHAPE_NAMES.append(f"{name}_accel")
for i in range(5):
    EXPANDED_BLENDSHAPE_NAMES.append(f"extra_temporal_feat_{i}")
BLENDSHAPE_NAMES = EXPANDED_BLENDSHAPE_NAMES + _RATIO_NAMES

# From WTgEksZf0mww
INPUT_DIM = 165
NUM_CLASSES = 5
L2_REG = 1e-4 # Defined here as it's used in build_fer_model

# --- Model Architecture (Copied from WTgEksZf0mww) ---
def glu_block(x, units: int, dropout_rate: float, name: str | None = None):
    # Gated Linear Unit for non-linearity
    gate = layers.Dense(units, activation="sigmoid", name=f"{name}_gate" if name else None)(x)
    linear = layers.Dense(units, name=f"{name}_linear" if name else None)(x)
    h = layers.Multiply()([gate, linear])
    h = layers.LayerNormalization(name=f"{name}_ln" if name else None)(h)
    h = layers.Dropout(dropout_rate)(h)
    return h

def semantic_attention_bottleneck(x, units: int = 224, num_heads: int = 4, name: str | None = None):
    # Simple Multi-Head Self-Attention for semantic tokenization
    # Takes the original 56 blendshapes and outputs a fixed-size embedding

    # Project to attention dimension
    q = layers.Dense(units, name=f"{name}_q" if name else None)(x)
    k = layers.Dense(units, name=f"{name}_k" if name else None)(x)
    v = layers.Dense(units, name=f"{name}_v" if name else None)(x)

    # Multi-head attention (simplified, not full Transformer block)
    attention_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=units // num_heads,
                                                name=f"{name}_mha" if name else None)(q, k, v)

    # Global average pooling to condense into a single vector (224-dim output)
    return layers.GlobalAveragePooling1D(name=f"{name}_gap" if name else None)(attention_output)


def residual_block(x, units: int, dropout_rate: float, name: str):
    h = layers.BatchNormalization(name=f"{name}_bn1")(x)
    h = layers.Activation("swish", name=f"{name}_swish1")(h)
    h = layers.Dense(units, use_bias=False, kernel_initializer="he_uniform",
                     kernel_regularizer=regularizers.l2(L2_REG))(h)
    h = layers.BatchNormalization(name=f"{name}_bn2")(h)
    h = layers.Activation("swish", name=f"{name}_swish2")(h)
    h = layers.Dropout(dropout_rate)(h)
    h = layers.Dense(units, use_bias=False, kernel_initializer="he_uniform",
                     kernel_regularizer=regularizers.l2(L2_REG))(h)

    if x.shape[-1] != units:
        x = layers.Dense(units, use_bias=False, kernel_initializer="he_uniform")(x)
    return layers.Add()([h, x])

def build_fer_model():
    # INPUT_DIM is now 165 because of the expansion above
    inp = layers.Input(shape=(165,), name="features_165d")

    # Stage 1: Noise Filtering via GLU
    x = layers.Dense(128)(inp)
    x = layers.LayerNormalization()(x)
    x = glu_block(x, 128, dropout_rate=0.25, name="glu1")

    # Stage 2: Attention (Groups features into semantic tokens)
    # We take the original 56 "core" features for the attention path
    core_features = layers.Lambda(lambda x: x[:, :56], name="core_features_extract")(inp)
    # Reshape for attention (batch, sequence_length=56, feature_dim=1) -> (batch, 56, 1)
    core_features_reshaped = layers.Reshape((56, 1), name="core_features_reshape")(core_features)
    attn_path = semantic_attention_bottleneck(core_features_reshaped, name="semantic_attn") # 224-dim

    # Stage 3: Fusion
    fused = layers.Concatenate(name="fused_features")([x, attn_path])
    fused = layers.Dense(128, activation='swish', name="embedding_layer")(fused)

    # Stage 4: Classification Head (Outputting Logits for Temperature Scaling)
    fused = glu_block(fused, 64, dropout_rate=0.2, name="glu2")
    out = layers.Dense(NUM_CLASSES, activation=None, name="predictions")(fused)

    return keras.Model(inputs=inp, outputs=out)
# --- End Model Architecture ---

# --- ADD: Categorical Focal Loss Class ---
class CategoricalFocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_pred = tf.nn.softmax(y_pred) # Convert logits to probs
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        # Fix: Use tf.pow instead of tf.math.power
        loss = -y_true * tf.math.log(y_pred) * tf.pow(1.0 - y_pred, self.gamma)
        return tf.reduce_sum(loss, axis=-1)

# --- ADD: SupervisedContrastiveLoss Class ---
class SupervisedContrastiveLoss(tf.keras.losses.Loss):
    def __init__(self, temperature=0.07, name=None):
        super().__init__(name=name)
        self.temperature = temperature

    def call(self, labels, feature_vectors):
        # Normalize feature vectors
        feature_vectors = tf.math.l2_normalize(feature_vectors, axis=1)

        # Compute pairwise dot product similarities
        # shape (batch_size, batch_size)
        similarity_matrix = tf.matmul(feature_vectors, feature_vectors, transpose_b=True)

        # Create a mask for positive pairs (samples from the same class)
        # Convert labels to integer type if they are not already
        labels = tf.cast(labels, dtype=tf.int32)
        labels_reshaped = tf.expand_dims(labels, -1)
        label_matrix = tf.cast(tf.equal(labels_reshaped, tf.transpose(labels_reshaped)), tf.float32)

        # Remove self-similarity from positive pairs
        positive_mask = label_matrix - tf.eye(tf.shape(labels)[0], dtype=tf.float32)

        # Create a mask to exclude self-similarity for the denominator (all other samples)
        identity_mask = tf.eye(tf.shape(labels)[0], dtype=tf.float32)
        negative_mask = 1.0 - identity_mask

        # Scale similarities by temperature
        similarity_matrix = similarity_matrix / self.temperature

        # Denominator: sum of exp(sim_ia/tau) for all a != i
        all_other_sim_sum = tf.reduce_sum(tf.exp(similarity_matrix) * negative_mask, axis=1)

        # Numerator: sum of exp(sim_ip/tau) for p in P(i)
        pos_sim_values_sum = tf.reduce_sum(tf.exp(similarity_matrix) * positive_mask, axis=1)

        # Compute log(numerator / denominator)
        log_term = tf.math.log(pos_sim_values_sum + 1e-9) - tf.math.log(all_other_sim_sum + 1e-9)

        # Divide by number of positive pairs for each anchor
        num_pos_pairs_per_anchor = tf.reduce_sum(positive_mask, axis=1)

        # Filter out anchors that have no positive pairs in the batch
        valid_anchors_mask = tf.greater(num_pos_pairs_per_anchor, 0)
        num_valid_anchors = tf.reduce_sum(tf.cast(valid_anchors_mask, tf.float32))

        # Use tf.cond for graph-compatible conditional logic
        def true_fn():
            return tf.constant(0.0, dtype=tf.float32)

        def false_fn():
            log_term_masked = tf.boolean_mask(log_term, valid_anchors_mask)
            num_pos_pairs_masked = tf.boolean_mask(num_pos_pairs_per_anchor, valid_anchors_mask)
            loss = -tf.reduce_sum(log_term_masked / num_pos_pairs_masked) / num_valid_anchors
            return loss

        return tf.cond(tf.equal(num_valid_anchors, 0), true_fn, false_fn)

# 1. Define the combined loss
def compute_loss(y_true_one_hot, y_true_int, logits, embeddings):
    # Focal Loss handles the class imbalance (Happy vs Sad)
    focal = CategoricalFocalLoss(gamma=2.0)(y_true_one_hot, logits)
    # SupCon pulls "Sad" embeddings together and pushes them away from "Neutral"
    supcon = SupervisedContrastiveLoss(temperature=0.07)(y_true_int, embeddings)
    return focal + (0.3 * supcon)

# ── 3.1 Load & Split ──────────────────────────────────────────────────────────
df = pd.read_csv(OUTPUT_CSV)

X = df[BLENDSHAPE_NAMES].values.astype(np.float32)
y = df["label"].values.astype(np.int32)

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.12, stratify=y_trainval, random_state=42
)

print(f"Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}")

# ── 3.2 Normalisation ─────────────────────────────────────────────────────────
# Blendshapes are in [0,1] but have different distributions per coefficient.
# StandardScaler ensures each muscle activation contributes equally to the loss.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# Save scaler — needed for inference
with open("blendshape_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
print("✅ Scaler saved → blendshape_scaler.pkl")

# ── 3.3 Class Imbalance: Oversample minority classes & 3.4 Class Weights for Loss ──────────────────────────

# Oversample all classes to be equal
ros = RandomOverSampler(sampling_strategy='not majority', random_state=42)
X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

# Calculate weights to help with Neutral vs Sad confusion
classes = np.unique(y_train_res)
weights = compute_class_weight('balanced', classes=classes, y=y_train_res)
class_weight_dict = dict(zip(classes, weights))

# Heavy penalty for misclassifying Sad (4) or Fear (1)
if 4 in class_weight_dict:
    class_weight_dict[4] *= 1.5
if 1 in class_weight_dict:
    class_weight_dict[1] *= 1.2

print("✅ Training on 5 balanced classes with IOD-stabilized features.")

# ── 3.5 MixUp Augmentation ────────────────────────────────────────────────────
def mixup_batch(X_batch, y_batch, alpha=0.3):
    """MixUp augmentation for blendshape feature vectors.
    Interpolates between random pairs in the batch.
    Particularly helps fear/surprise confusion boundary."""
    batch_size = len(X_batch)
    lam = np.random.beta(alpha, alpha, batch_size).astype(np.float32)
    lam = np.maximum(lam, 1 - lam)  # always >= 0.5 to preserve label

    perm = np.random.permutation(batch_size)
    X_mix = (lam[:, None] * X_batch) + ((1 - lam[:, None]) * X_batch[perm])

    # Soft labels (works with label_smoothing in CategoricalCrossentropy)
    y_oh  = tf.one_hot(y_batch, NUM_CLASSES).numpy()
    y_mix = (lam[:, None] * y_oh) + ((1 - lam[:, None]) * y_oh[perm])
    return X_mix, y_mix


def make_mixup_dataset(X, y, batch_size=512, shuffle=True, use_mixup=True):
    """Creates a tf.data.Dataset with optional MixUp."""
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(X), seed=42)
    ds = ds.batch(batch_size, drop_remainder=use_mixup) # drop_remainder for SupCon

    if use_mixup:
        def apply_mixup(X_b, y_b):
            X_m, y_m = tf.py_function(
                func=lambda x, y: mixup_batch(x.numpy(), y.numpy()),
                inp=[X_b, y_b],
                Tout=[tf.float32, tf.float32]
            )
            X_m.set_shape([None, INPUT_DIM]) # Ensure shape matches new INPUT_DIM
            y_m.set_shape([None, NUM_CLASSES])
            return X_m, y_m
        ds = ds.map(apply_mixup, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.map(
            lambda x, y: (x, tf.one_hot(y, NUM_CLASSES)),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    return ds.prefetch(tf.data.AUTOTUNE)


BATCH_SIZE = 512
train_ds = make_mixup_dataset(X_train_res, y_train_res, BATCH_SIZE, shuffle=True,  use_mixup=True)
val_ds   = make_mixup_dataset(X_val,       y_val,       BATCH_SIZE, shuffle=False, use_mixup=False)

# ── 3.6 Setup Model and Optimizer ───────────────────────────────────────────────
model = build_fer_model()

# Define the sub-model for embeddings ONCE here, not in the loop
embedding_layer_output = model.get_layer("embedding_layer").output
embedding_model = tf.keras.Model(inputs=model.input, outputs=embedding_layer_output)

# --- UPDATE: Learning Rate Schedule (Cosine Decay with Warmup) ---
lr_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=3e-4,
    first_decay_steps=1000,
    t_mul=2.0,
    m_mul=0.9,
    alpha=1e-5
)

optimizer = keras.optimizers.AdamW(
    learning_rate=lr_schedule,
    weight_decay=1e-5,
    clipnorm=1.0,   # gradient clipping for training stability
)

# --- Custom Training Loop Setup (replaces model.compile and model.fit) ---

# Define metrics in a class for better management
class MetricTracker:
    def __init__(self):
        self.train_loss = tf.keras.metrics.Mean(name="train_loss")
        self.train_accuracy = tf.keras.metrics.CategoricalAccuracy(name="train_accuracy")
        self.train_auc = tf.keras.metrics.AUC(name="train_auc", multi_label=False)

        self.val_loss = tf.keras.metrics.Mean(name="val_loss")
        self.val_accuracy = tf.keras.metrics.CategoricalAccuracy(name="val_accuracy")
        self.val_auc = tf.keras.metrics.AUC(name="val_auc", multi_label=False)

    def reset_states(self):
        self.train_loss.reset_state()
        self.train_accuracy.reset_state()
        self.train_auc.reset_state()
        self.val_loss.reset_state()
        self.val_accuracy.reset_state()
        self.val_auc.reset_state()

# Instantiate the metric tracker
metric_tracker = MetricTracker()

# 2. Add this to your model.compile or custom loop
@tf.function
def train_step(data):
    x, y_one_hot = data
    y_int = tf.argmax(y_one_hot, axis=-1, output_type=tf.int32) # Convert one-hot to integer labels

    with tf.GradientTape() as tape:
        # Get both the final prediction and the internal embedding
        logits = model(x, training=True)
        # Use the pre-defined model here
        embeddings = embedding_model(x, training=True)

        loss = compute_loss(y_one_hot, y_int, logits, embeddings)

    # Standard backprop...
    trainable_vars = model.trainable_variables
    gradients = tape.gradient(loss, trainable_vars)
    optimizer.apply_gradients(zip(gradients, trainable_vars))

    # Update metrics using the tracker instance
    metric_tracker.train_loss.update_state(loss)
    metric_tracker.train_accuracy.update_state(y_one_hot, logits)
    metric_tracker.train_auc.update_state(y_one_hot, logits)

@tf.function
def val_step(data):
    x, y_one_hot = data
    y_int = tf.argmax(y_one_hot, axis=-1, output_type=tf.int32)

    logits = model(x, training=False)
    embeddings = embedding_model(x, training=False)

    loss = compute_loss(y_one_hot, y_int, logits, embeddings)

    # Update metrics using the tracker instance
    metric_tracker.val_loss.update_state(loss)
    metric_tracker.val_accuracy.update_state(y_one_hot, logits)
    metric_tracker.val_auc.update_state(y_one_hot, logits)


# ── 3.7 Callbacks ─────────────────────────────────────────────────────────────
# Callbacks for a custom training loop typically need manual integration.
# For simplicity, we'll keep the ModelCheckpoint for saving the best model
# and manually implement early stopping and LR reduction logic.

# Original callbacks (commented out as they are designed for model.fit)
# callbacks = [
#     # Reduce LR on plateau — monitors val_loss, not val_accuracy
#     # (val_loss is more stable with label smoothing)
#     keras.callbacks.ReduceLROnPlateau(
#         monitor="val_loss",
#         factor=0.4,
#         patience=5,
#         min_lr=1e-6,
#         verbose=1,
#     ),
#     # Early stopping with val_accuracy (the actual metric we care about)
#     keras.callbacks.EarlyStopping(
#         monitor="val_accuracy",
#         patience=12,
#         restore_best_weights=True,
#         verbose=1,
#         mode="max",
#     ),
#     # Save best checkpoint
#     keras.callbacks.ModelCheckpoint(
#         filepath="best_fer_model.keras",
#         monitor="val_accuracy",
#         save_best_only=True,
#         verbose=0,
#         mode="max",
#     ),
#     # TensorBoard (optional)
#     keras.callbacks.TensorBoard(
#         log_dir="./logs", histogram_freq=0, write_graph=False
#     ),
# ]

# ── 3.8 Train (Custom Loop) ───────────────────────────────────────────────────
from tqdm.auto import tqdm

print("\n🚀 Starting optimized training loop...")

EPOCHS = 150 # Max epochs
BEST_VAL_ACCURACY = 0
PATIENCE_COUNTER = 0
EARLY_STOP_PATIENCE = 12 # From original EarlyStopping callback

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    # Reset metrics for the new epoch using the tracker
    metric_tracker.reset_states()

    # Training loop with progress bar
    step_pb = tqdm(enumerate(train_ds), total=len(train_ds), desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for step, data in step_pb:
        train_step(data)
        # Optional: update progress bar with current loss
        if step % 10 == 0:
            step_pb.set_postfix({'loss': f"{metric_tracker.train_loss.result():.4f}"})

    # Validation loop with progress bar
    val_pb = tqdm(enumerate(val_ds), total=len(val_ds), desc=f"Epoch {epoch+1}/{EPOCHS} [Val]", leave=False)
    for step, data in val_pb:
        val_step(data)

    # Print progress (access results from the tracker)
    train_loss = metric_tracker.train_loss.result()
    train_accuracy = metric_tracker.train_accuracy.result()
    train_auc = metric_tracker.train_auc.result()
    val_loss = metric_tracker.val_loss.result()
    val_accuracy = metric_tracker.val_accuracy.result()
    val_auc = metric_tracker.val_auc.result()
    current_lr = optimizer.learning_rate.numpy()

    print(f"  LR: {current_lr:.6f} - "
          f"Train Loss: {train_loss:.4f} - Train Acc: {train_accuracy:.4f} - Train AUC: {train_auc:.4f} - "
          f"Val Loss: {val_loss:.4f} - Val Acc: {val_accuracy:.4f} - Val AUC: {val_auc:.4f}")

    # Manual ModelCheckpoint and EarlyStopping logic
    if val_accuracy > BEST_VAL_ACCURACY:
        BEST_VAL_ACCURACY = val_accuracy
        PATIENCE_COUNTER = 0
        model.save("best_fer_model.keras") # Save the best model
        print("  ✅ Model saved: best_fer_model.keras")
    else:
        PATIENCE_COUNTER += 1
        print(f"  Patience counter: {PATIENCE_COUNTER}/{EARLY_STOP_PATIENCE}")
        if PATIENCE_COUNTER >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping triggered after {epoch + 1} epochs.")
            model.load_weights("best_fer_model.keras") # Restore best weights
            break


# ── 3.9 Final evaluation ──────────────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix

y_pred_prob = model.predict(X_test, batch_size=1024, verbose=0)
y_pred      = np.argmax(y_pred_prob, axis=1)

EMOTION_NAMES = ["Anger", "Fear", "Happy", "Neutral", "Sad"]
print("\n📊 Test Set Results")
print(classification_report(y_test, y_pred, target_names=EMOTION_NAMES, digits=4))

cm = confusion_matrix(y_test, y_pred, normalize='true')
print("Normalised confusion matrix (row=true, col=pred):")
print(pd.DataFrame(cm.round(3), index=EMOTION_NAMES, columns=EMOTION_NAMES).to_string())

Train: 1,210  Val: 166  Test: 243
✅ Scaler saved → blendshape_scaler.pkl
✅ Training on 5 balanced classes with IOD-stabilized features.

🚀 Starting optimized training loop...

Epoch 1/150


Epoch 1/150 [Train]:   0%|          | 0/2 [00:00<?, ?it/s]

KeyboardInterrupt: 

---
## BLOCK 4 — TFLite Export with Full INT8 Quantization

### Why INT8?
- **4× size reduction** vs FP32 (~310KB vs ~1.2MB)
- **2–4× faster** on mobile NPU (Hexagon DSP, Apple Neural Engine, etc.)
- **Accuracy drop typically <0.5%** on a well-trained model

### Representative Dataset for Calibration
INT8 quantization requires a "representative dataset" to calibrate
activation ranges. We use the actual validation blendshapes — this
ensures the quantization range captures the real data distribution.

In [ ]:
import tensorflow as tf
import json
import numpy as np # Explicitly import numpy

model_path="/content/finetuned_fer_model.keras"

# Load the Keras model object, disabling safe_mode for Lambda layers
kera_model = tf.keras.models.load_model(model_path, compile=False, safe_mode=False)

# Get X_train_res and INPUT_DIM from the global scope
# This assumes previous cells (like yPMVKChB0mw0 and WTgEksZf0mww) have been executed.
try:
    _X_train_res = globals()['X_train_res']
    _INPUT_DIM = globals()['INPUT_DIM']
except KeyError as e:
    raise NameError(
        f"Required variable '{e.args[0]}' is not defined. "
        "Please ensure all preceding data preparation and model definition cells "
        "have been executed successfully before running this cell."
    )

# 1. Prepare Representative Dataset for Calibration
# We use a slice of the scaled training data so the quantizer knows the dynamic range
def representative_data_gen():
    # Use 500-1000 samples for high-precision calibration
    # Access _X_train_res and _INPUT_DIM which are bound in this closure
    for i in range(min(1000, len(_X_train_res))):
        sample = _X_train_res[i].astype(np.float32).reshape(1, _INPUT_DIM)
        yield [sample]

# 2. Configure TFLite Converter
converter = tf.lite.TFLiteConverter.from_keras_model(kera_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen

# Ensure full integer quantization
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# 3. Convert and Save Model
tflite_model_int8 = converter.convert()

with open("fer_blendshape_int8.tflite", "wb") as f:
    f.write(tflite_model_int8)

print("✅ Model converted to Full INT8 Quantization.")

# 4. Extract Quantization Parameters for the App Metadata
interpreter = tf.lite.Interpreter(model_content=tflite_model_int8)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

# These are required for the 'Quantization Sandwich' logic in your app
inp_scale, inp_zero = input_details['quantization']
out_scale, out_zero = output_details['quantization']

# 5. Save Comprehensive Metadata
# This JSON is the "bridge" between your Python training and Mobile inference
metadata = {
    "mean": scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist(),
    "feature_names": BLENDSHAPE_NAMES,
    "emotion_labels": EMOTION_NAMES,
    "inp_quantization": {"scale": float(inp_scale), "zero_point": int(inp_zero)},
    "out_quantization": {"scale": float(out_scale), "zero_point": int(out_zero)},
    "model_type": "5-class_residual_mlp_int8"
}

with open("blendshape_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("✅ blendshape_metadata.json updated with new 5-class normalization constants.")

Saved artifact at '/tmp/tmpjpwd8ap1'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 165), dtype=tf.float32, name='features_165d')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  134106934842320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134106934843280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134106934844432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134106934843856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134106934844048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134106934843472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134106934845008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134106934843664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134106934845392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134106934845200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134106934845776: Tens

---
## BLOCK 5 — Real-Time Inference with Temporal Smoothing

### The Flickering Problem
Without smoothing, frame-by-frame predictions oscillate rapidly (e.g.,
`happy → neutral → happy`) because:
1. Blendshapes vary slightly frame-to-frame (natural micro-expressions)
2. The model's softmax output changes for near-boundary samples

### Solutions Implemented
| Strategy | Latency Added | Best For |
|----------|--------------|----------|
| Simple Moving Average | 0ms (pure buffer math) | Probability smoothing |
| Exponential Moving Average | 0ms | Prioritises recent frames |
| Majority Vote (argmax) | 0ms | Label smoothing for display |
| Kalman Filter | ~0.1ms | Prediction with velocity (advanced) |

In [ ]:
# ── BLOCK 5: Temporal Smoothing + Full Inference Pipeline ─────────────────────

from collections import deque
import time
import numpy as np
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
import json
import tensorflow as tf

# Assuming TemporalFeatureExtractor and calculate_ratios are defined in an earlier cell
# and are accessible in this scope (e.g., from N8GRF7MQ0mws)
# If not, they would need to be re-defined or explicitly imported.


class TemporalSmoother:
    """
    Smooths per-frame emotion probability vectors to prevent display flickering.

    Usage:
        smoother = TemporalSmoother(window=7, strategy='ema', ema_alpha=0.35)
        for frame in video_stream:
            raw_probs = model.predict(frame)   # shape: (7,)
            smoothed  = smoother.update(raw_probs)
            label     = smoother.get_label()
    """

    STRATEGIES = ('sma', 'ema', 'majority_vote')

    def __init__(
        self,
        window: int   = 3,
        strategy: str = 'ema',
        ema_alpha: float = 0.35,
        num_classes: int = NUM_CLASSES,
        emotion_names: list = None,
    ):
        """
        Args:
            window:     Number of frames to consider. 5-10 is sweet-spot for 30fps.
            strategy:   'sma' (simple moving avg), 'ema' or
                        'majority_vote' (most common argmax in window).
            ema_alpha:  EMA decay factor. Higher = reacts faster. Range (0,1).
                        0.35 \u2248 5-frame effective window at 30fps.
        """
        assert strategy in self.STRATEGIES, f"Unknown strategy: {strategy}"
        self.window      = window
        self.strategy    = strategy
        self.alpha       = ema_alpha
        self.num_classes = num_classes
        self.names       = emotion_names or [str(i) for i in range(num_classes)]

        # Circular buffer of raw probability vectors
        self._prob_buffer = deque(maxlen=window)
        # EMA state
        self._ema_state   = np.ones(num_classes, dtype=np.float32) / num_classes
        # Label vote buffer (argmax per frame)
        self._vote_buffer = deque(maxlen=window)
        # Output
        self.smoothed_probs  = np.ones(num_classes, dtype=np.float32) / num_classes
        self.current_label   = -1
        self.confidence      = 0.0

    def update(self, raw_probs: np.ndarray) -> np.ndarray:
        """
        Ingest a new per-frame probability vector.

        Args:
            raw_probs: shape (num_classes,), values in [0, 1], sum \u2248 1.
                       Accepts INT8 dequantised output directly.
        Returns:
            smoothed_probs: shape (num_classes,)
        """
        probs = np.asarray(raw_probs, dtype=np.float32).flatten()
        # Re-normalise in case of dequantisation rounding
        probs = probs / (probs.sum() + 1e-9)

        self._prob_buffer.append(probs)
        self._vote_buffer.append(int(np.argmax(probs)))

        if self.strategy == 'sma':
            self.smoothed_probs = np.mean(list(self._prob_buffer), axis=0)

        elif self.strategy == 'ema':
            # Exponential moving average: p_t = \u03b1\u00b7raw + (1-\u03b1)\u00b7p_{t-1}
            self._ema_state    = self.alpha * probs + (1 - self.alpha) * self._ema_state
            self.smoothed_probs = self._ema_state / (self._ema_state.sum() + 1e-9)

        elif self.strategy == 'majority_vote':
            # Most common argmax in the window, backed by average prob for ties
            votes  = list(self._vote_buffer)
            winner = max(set(votes), key=votes.count)
            # Build smoothed_probs: winner gets its average prob from buffer,
            # rest get uniform remainder
            avg_probs = np.mean(list(self._prob_buffer), axis=0)
            mask = np.zeros(self.num_classes, dtype=np.float32)
            mask[winner] = avg_probs[winner]
            remainder = (1 - mask[winner]) / (self.num_classes - 1 + 1e-9)
            for i in range(self.num_classes):
                if i != winner:
                    mask[i] = remainder
            self.smoothed_probs = mask

        self.current_label = int(np.argmax(self.smoothed_probs))
        self.confidence    = float(self.smoothed_probs[self.current_label])
        return self.smoothed_probs

    def get_label(self) -> str:
        """Returns the current smoothed emotion label string."""
        return self.names[self.current_label] if self.current_label >= 0 else "unknown"

    def get_top2(self) -> list[dict]:
        """Returns top-2 emotions with name and confidence."""
        top2_idx = np.argsort(self.smoothed_probs)[::-1][:2]
        return [{"emotion": self.names[i], "confidence": float(self.smoothed_probs[i])}
                for i in top2_idx]

    def reset(self):
        """Reset all buffers (e.g., on new face / scene cut)."""
        self._prob_buffer.clear()
        self._vote_buffer.clear()
        self._ema_state = np.ones(self.num_classes, dtype=np.float32) / self.num_classes


# ── Full Real-Time Inference Pipeline ─────────────────────────────────────────
class FERInferencePipeline:
    """
    End-to-end pipeline: image \u2192 MediaPipe blendshapes \u2192 normalise \u2192
    INT8 TFLite \u2192 temporal smooth \u2192 emotion label.

    Designed for frame-by-frame call from a camera loop.
    """
    HR_MAP = {0: "Unconfident", 1: "Unconfident", 2: "Happy", 3: "Neutral", 4: "Unconfident"}

    def __init__(
        self,
        tflite_path: str,
        metadata_path: str,
        landmarker_path: str,
        smoother_strategy: str = 'ema',
        smoother_window: int = 3,
        confidence_threshold: float = 0.45,
    ):
        # Load TFLite model
        self.interpreter = tf.lite.Interpreter(model_path=tflite_path)
        self.interpreter.allocate_tensors()
        self._inp = self.interpreter.get_input_details()[0]
        self._out = self.interpreter.get_output_details()[0]
        self._inp_scale, self._inp_zero = self._inp["quantization"]
        self._out_scale, self._out_zero = self._out["quantization"]

        # Load normalisation params
        with open(metadata_path) as f:
            meta = json.load(f)
        self.mean        = np.array(meta["mean"],  dtype=np.float32)
        self.scale       = np.array(meta["scale"], dtype=np.float32)
        self.emotion_names = meta["label_names"]

        # Build MediaPipe landmarker
        self._landmarker = build_landmarker(landmarker_path)

        # Temporal feature extractor
        self.temporal_ext = TemporalFeatureExtractor(window_size=smoother_window) # Added

        # Temporal smoother
        self.smoother = TemporalSmoother(
            window=smoother_window,
            strategy=smoother_strategy,
            num_classes=len(self.emotion_names),
            emotion_names=self.emotion_names,
        )
        self.confidence_threshold = confidence_threshold
        self._frame_times = deque(maxlen=30)  # for FPS tracking

        # Baseline features for calibration
        self.baseline = None
        self.is_calibrating = False
        self._calibration_buffer = []
        self.baseline_frames_needed = 0 # Will be set by calibrate_baseline

    def _extract(self, bgr_frame: np.ndarray) -> np.ndarray | None:
        """
        Internal helper to extract blendshape and ratio features from a BGR frame.
        Returns a (56,) float32 array or None if no face is detected.
        """
        rgb = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        det = self._landmarker.detect(mp_img)

        if det.face_blendshapes and det.face_landmarks:
            blendshape_scores = [bs.score for bs in det.face_blendshapes[0]]
            geometric_ratios = calculate_ratios(det.face_landmarks[0])
            return np.array(blendshape_scores + geometric_ratios, dtype=np.float32)
        return None


    def calibrate_baseline(self, num_frames=30):
        """Call this when the user is prompted to look 'Neutral' for 1-2 seconds."""
        self.is_calibrating = True
        self._calibration_buffer = []
        self.baseline_frames_needed = num_frames
        print("Starting calibration...")

    def _run_tflite_logits(self, x_norm: np.ndarray) -> np.ndarray:
        """Quantize, invoke TFLite, and dequantize output to get raw logits."""
        x_q = (x_norm / self._inp_scale + self._inp_zero).astype(np.int8)
        self.interpreter.set_tensor(self._inp["index"], x_q.reshape(1, -1))
        self.interpreter.invoke()
        y_q = self.interpreter.get_tensor(self._out["index"])
        logits = (y_q.astype(np.float32) - self._out_zero) * self._out_scale
        return logits[0]

    def run_tflite_with_temp_scaling(self, x_norm: np.ndarray, T: float = 1.0) -> np.ndarray:
        """Quantize, invoke TFLite, dequantize to logits, apply temperature scaling, and softmax."""
        logits = self._run_tflite_logits(x_norm)
        scaled_logits = logits / T
        probs = tf.nn.softmax(scaled_logits).numpy()
        return probs


    def predict_frame(self, bgr_frame: np.ndarray):
        t0 = time.perf_counter()

        # 1. Extract raw 56
        raw_56 = self._extract(bgr_frame)
        if raw_56 is None:
            self.smoother.reset()
            return "No Face"

        # 2. NEW: Temporal Expansion (Crucial for Sadness detection)
        # Assuming the first 52 features are blendshapes, and last 4 are ratios
        expanded_161 = self.temporal_ext.extract(raw_56[:52])
        expanded_165 = np.concatenate([expanded_161, raw_56[52:]])

        # 3. Normalize
        x_norm = (expanded_165 - self.mean) / self.scale

        # 4. Predict with Temperature Scaling (0.8051)
        probs = self.run_tflite_with_temp_scaling(x_norm, T=0.8051)

        # 5. Smooth and Map
        smoothed = self.smoother.update(probs)
        raw_idx = np.argmax(smoothed)

        # 6. Final UI Label
        display_label = self.HR_MAP[raw_idx]
        if smoothed[raw_idx] < self.confidence_threshold: display_label = "Uncertain"

        return display_label

    def close(self):
        self._landmarker.close()


# ── Demo: Simulate temporal smoothing on test data ─────────────────────────────
print("\u25b6\ufe0f Temporal Smoothing Demo (simulated 30fps stream)")
print("=" * 55)

# Updated to reflect 5 emotion classes: Anger, Fear, Happy, Neutral, Sad
demo_probs_sequence = [
    [0.05, 0.02, 0.82, 0.08, 0.03],  # Happy dominant
    [0.08, 0.05, 0.61, 0.16, 0.10],  # Happy dominant
    [0.10, 0.07, 0.58, 0.15, 0.10],  # Happy dominant
    [0.06, 0.04, 0.71, 0.12, 0.07],  # Happy dominant
    [0.04, 0.03, 0.79, 0.09, 0.05],  # Happy dominant
]

for strategy in ['sma', 'ema', 'majority_vote']:
    smoother = TemporalSmoother(
        window=5, strategy=strategy, emotion_names=EMOTION_NAMES
    )
    print(f"\n  Strategy: {strategy.upper()}")
    print(f"  {'Frame':<7} {'Raw Label':<12} {'Smooth Label':<15} {'Confidence':<12}")
    for i, probs in enumerate(demo_probs_sequence):
        raw_label = EMOTION_NAMES[np.argmax(probs)]
        smoother.update(np.array(probs))
        print(f"  {i+1:<7} {raw_label:<12} {smoother.get_label():<15} {smoother.confidence:.3f}")


▶️ Temporal Smoothing Demo (simulated 30fps stream)

  Strategy: SMA
  Frame   Raw Label    Smooth Label    Confidence  
  1       Happy        Happy           0.820
  2       Happy        Happy           0.715
  3       Happy        Happy           0.670
  4       Happy        Happy           0.680
  5       Happy        Happy           0.702

  Strategy: EMA
  Frame   Raw Label    Smooth Label    Confidence  
  1       Happy        Happy           0.417
  2       Happy        Happy           0.485
  3       Happy        Happy           0.518
  4       Happy        Happy           0.585
  5       Happy        Happy           0.657

  Strategy: MAJORITY_VOTE
  Frame   Raw Label    Smooth Label    Confidence  
  1       Happy        Happy           0.820
  2       Happy        Happy           0.715
  3       Happy        Happy           0.670
  4       Happy        Happy           0.680
  5       Happy        Happy           0.702


---
## BLOCK 6 — Benchmark & Validation

Quick sanity checks before mobile deployment.

In [ ]:
# ── BLOCK 6: Benchmark ────────────────────────────────────────────────────────
import time

print("⚡ Inference Latency Benchmark")
print("  (Simulates blendshape-only MLP inference — excludes MediaPipe time)")
print()

N_WARMUP = 50
N_BENCH  = 1000
sample   = X_test[0:1].astype(np.float32)

# Warmup
for _ in range(N_WARMUP):
    _ = model.predict(sample, verbose=0)

# FP32 Keras
t0 = time.perf_counter()
for _ in range(N_BENCH):
    model.predict(sample, verbose=0)
fp32_ms = (time.perf_counter() - t0) / N_BENCH * 1000

# INT8 TFLite
x_q = (sample / inp_scale + inp_zero).astype(np.int8).reshape(1, INPUT_DIM)
t0 = time.perf_counter()
for _ in range(N_BENCH):
    interpreter.set_tensor(inp_details["index"], x_q)
    interpreter.invoke()
    interpreter.get_tensor(out_details["index"])
int8_ms = (time.perf_counter() - t0) / N_BENCH * 1000

print(f"  FP32 Keras  : {fp32_ms:.3f} ms/frame")
print(f"  INT8 TFLite : {int8_ms:.3f} ms/frame  ({fp32_ms/int8_ms:.1f}× speedup)")
print(f"  Budget left : {1000/30 - int8_ms:.1f} ms for MediaPipe + display (at 30fps)")
print()

# File size summary
print("📦 Artifact Sizes")
for path in ["fer_blendshape_int8.tflite", "best_fer_model.keras", "blendshape_scaler.pkl"]:
    if Path(path).exists():
        kb = Path(path).stat().st_size / 1024
        status = "✅" if kb < 5120 else "❌ OVER 5MB!"
        print(f"  {status} {path:<35} {kb:>8.1f} KB")

print()
print("✅ Pipeline complete. Bundle these files for mobile deployment:")
print("   • fer_blendshape_int8.tflite   — the model")
print("   • blendshape_metadata.json    — scaler + quantisation params")
print("   • face_landmarker.task        — MediaPipe face detector")

⚡ Inference Latency Benchmark
  (Simulates blendshape-only MLP inference — excludes MediaPipe time)

  FP32 Keras  : 115.759 ms/frame
  INT8 TFLite : 0.119 ms/frame  (969.6× speedup)
  Budget left : 33.2 ms for MediaPipe + display (at 30fps)

📦 Artifact Sizes
  ✅ fer_blendshape_int8.tflite             392.1 KB
  ✅ best_fer_model.keras                  4379.5 KB
  ✅ blendshape_scaler.pkl                    1.8 KB

✅ Pipeline complete. Bundle these files for mobile deployment:
   • fer_blendshape_int8.tflite   — the model
   • blendshape_metadata.json    — scaler + quantisation params
   • face_landmarker.task        — MediaPipe face detector


In [ ]:
import scipy.optimize

def temperature_scaling(logits, temp):
    return logits / temp

def calibration_objective(temp, logits, labels):
    scaled_logits = temperature_scaling(logits, temp)
    loss = tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(labels=labels, logits=scaled_logits)
    )
    return loss.numpy()

# 1. Get raw logits from val set (Requires model output without softmax)
# Note: You can also use log(softmax_output) as an approximation
val_probs = model.predict(X_val)
val_logits = np.log(val_probs + 1e-9)

# 2. Find optimal T (T > 1 usually 'softens' overconfident predictions)
res = scipy.optimize.minimize(
    calibration_objective, x0=1.0, args=(val_logits, y_val),
    method='L-BFGS-B', bounds=[(0.1, 5.0)]
)

optimal_temp = res.x[0]
print(f"🔥 Optimal Calibration Temperature: {optimal_temp:.4f}")

# 3. Add to your blendshape_metadata.json for Mobile Inference
# In Flutter: Adjusted_Probs = Softmax(Logits / optimal_temp)


80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
🔥 Optimal Calibration Temperature: 0.8051


In [ ]:
import tensorflow as tf
import json
import numpy as np # Explicitly import numpy

model_path="/content/finetuned_fer_model.keras"

# Load the Keras model object, disabling safe_mode for Lambda layers
kera_model = tf.keras.models.load_model(model_path, compile=False, safe_mode=False)

# Get X_train_res and INPUT_DIM from the global scope
# This assumes previous cells (like yPMVKChB0mw0 and WTgEksZf0mww) have been executed.
try:
    _X_train_res = globals()['X_train_res']
    _INPUT_DIM = globals()['INPUT_DIM']
except KeyError as e:
    raise NameError(
        f"Required variable '{e.args[0]}' is not defined. "
        "Please ensure all preceding data preparation and model definition cells "
        "have been executed successfully before running this cell."
    )

# 1. Prepare Representative Dataset for Calibration
# We use a slice of the scaled training data so the quantizer knows the dynamic range
def representative_data_gen():
    # Use 500-1000 samples for high-precision calibration
    # Access _X_train_res and _INPUT_DIM which are bound in this closure
    for i in range(min(1000, len(_X_train_res))):
        sample = _X_train_res[i].astype(np.float32).reshape(1, _INPUT_DIM)
        yield [sample]

# 2. Configure TFLite Converter
converter = tf.lite.TFLiteConverter.from_keras_model(kera_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen

# Ensure full integer quantization
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# 3. Convert and Save Model
tflite_model_int8 = converter.convert()

with open("fer_blendshape_int8.tflite", "wb") as f:
    f.write(tflite_model_int8)

print("✅ Model converted to Full INT8 Quantization.")

# 4. Extract Quantization Parameters for the App Metadata
interpreter = tf.lite.Interpreter(model_content=tflite_model_int8)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

# These are required for the 'Quantization Sandwich' logic in your app
inp_scale, inp_zero = input_details['quantization']
out_scale, out_zero = output_details['quantization']

# 5. Save Comprehensive Metadata
# This JSON is the "bridge" between your Python training and Mobile inference
metadata = {
    "mean": scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist(),
    "feature_names": BLENDSHAPE_NAMES,
    "emotion_labels": EMOTION_NAMES,
    "inp_quantization": {"scale": float(inp_scale), "zero_point": int(inp_zero)},
    "out_quantization": {"scale": float(out_scale), "zero_point": int(out_zero)},
    "model_type": "5-class_residual_mlp_int8"
}

with open("blendshape_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("✅ blendshape_metadata.json updated with new 5-class normalization constants.")

NameError: Required variable 'X_train_res' is not defined. Please ensure all preceding data preparation and model definition cells have been executed successfully before running this cell.

In [ ]:
#!C:/Users/white/AppData/Local/Programs/Python/Python311/python.exe
"""
RAF-DB Evaluation Script — FER Blendshape Pipeline (INT8 TFLite)
=================================================================
Senior ML Engineer Review: Rigorous evaluation covering:
  1. Feature extraction parity check (index-order guard)
  2. INT8 quantisation vs FP32 accuracy delta
  3. Head-pose stability / "face lift" regression test
  4. Per-class Precision/Recall for the 5-class model
  5. Mapped 3-class "Virtual HR" Precision/Recall
  6. Normalised confusion matrices (both label spaces)
  7. Temporal smoothing (EMA) accuracy simulation on image sequences

Usage:
    python evaluate_rafdb.py \
        --rafdb_root  ./testing rafdb \
        --tflite      ./fer_blendshape_int8.tflite \
        --metadata    ./blendshape_metadata.json \
        --landmarker  ./face_landmarker.task \
        --fp32_keras  ./best_fer_model.keras   # optional – enables quant delta
        --output_dir  ./eval_results

Directory layout expected for --rafdb_root:
    basic/
      test/
        Anger/  Fear/  Happy/  Neutral/  Sad/
      train/    (not used here, but must be present)

Requirements:
    pip install mediapipe>=0.10.9 tensorflow>=2.15 scikit-learn opencv-python \
                tqdm matplotlib seaborn pandas numpy
"""

import argparse
import json
import math
import os
import sys
import time
import warnings
from collections import deque, defaultdict
from pathlib import Path
from typing import Any

warnings.filterwarnings("ignore")

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import tensorflow as tf
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from tqdm import tqdm

# ─────────────────────────────────────────────────────────────────────────────
# 0.  CONSTANTS  (must be identical to training — this is your leakage guard)
# ─────────────────────────────────────────────────────────────────────────────

# ── Canonical blendshape ORDER (52 names, index 0-51). ────────────────────────
# Any deviation between training extraction and here causes silent feature drift.
CANONICAL_BLENDSHAPE_NAMES: list[str] = [
    "_neutral",
    "browDownLeft", "browDownRight", "browInnerUp",
    "browOuterUpLeft", "browOuterUpRight",
    "cheekPuff", "cheekSquintLeft", "cheekSquintRight",
    "eyeBlinkLeft", "eyeBlinkRight",
    "eyeLookDownLeft", "eyeLookDownRight",
    "eyeLookInLeft", "eyeLookInRight",
    "eyeLookOutLeft", "eyeLookOutRight",
    "eyeLookUpLeft", "eyeLookUpRight",
    "eyeSquintLeft", "eyeSquintRight",
    "eyeWideLeft", "eyeWideRight",
    "jawForward", "jawLeft", "jawOpen", "jawRight",
    "mouthClose",
    "mouthDimpleLeft", "mouthDimpleRight",
    "mouthFrownLeft", "mouthFrownRight",
    "mouthFunnel",
    "mouthLeft", "mouthLowerDownLeft", "mouthLowerDownRight",
    "mouthPressLeft", "mouthPressRight",
    "mouthPucker", "mouthRight",
    "mouthRollLower", "mouthRollUpper",
    "mouthShrugLower", "mouthShrugUpper",
    "mouthSmileLeft", "mouthSmileRight",
    "mouthStretchLeft", "mouthStretchRight",
    "mouthUpperUpLeft", "mouthUpperUpRight",
    "noseSneerLeft", "noseSneerRight",
]  # 52 items

# Appended geometric ratios (indices 52-55)
RATIO_NAMES: list[str] = ["ear", "mar", "brow_to_eye_dist", "mouth_pull"]

# Temporal expansion names from training:
# 52 current + 52 delta + 52 accel + 5 extra temporal + 4 ratios = 165
EXPANDED_BLENDSHAPE_NAMES: list[str] = []
for _name in CANONICAL_BLENDSHAPE_NAMES:
    EXPANDED_BLENDSHAPE_NAMES.append(f"{_name}_current")
    EXPANDED_BLENDSHAPE_NAMES.append(f"{_name}_delta")
    EXPANDED_BLENDSHAPE_NAMES.append(f"{_name}_accel")
for _i in range(5):
    EXPANDED_BLENDSHAPE_NAMES.append(f"extra_temporal_feat_{_i}")

# Full 165-dim feature vector (must match training CSV/scaler order exactly)
FEATURE_NAMES: list[str] = EXPANDED_BLENDSHAPE_NAMES + RATIO_NAMES
INPUT_DIM: int = 165

# Class definitions
EMOTION_LABELS: dict[str, int] = {
    "Anger": 0, "Fear": 1, "Happy": 2, "Neutral": 3, "Sad": 4,
}
EMOTION_NAMES: list[str] = ["Anger", "Fear", "Happy", "Neutral", "Sad"]
NUM_CLASSES: int = 5

# Virtual HR mapping: fine-grain → 3-class
HR_MAP: dict[int, int] = {0: 0, 1: 0, 2: 1, 3: 2, 4: 0}
HR_NAMES: list[str] = ["Unconfident", "Happy", "Neutral"]

# Landmark indices for ratio calculation
INNER_LEFT_EYE_IDX = 133   # inner corner left eye
INNER_RIGHT_EYE_IDX = 362  # inner corner right eye

IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


# ─────────────────────────────────────────────────────────────────────────────
# 1.  FEATURE EXTRACTION  (exact mirror of training Block 1)
# ─────────────────────────────────────────────────────────────────────────────

class TemporalFeatureExtractor:
    """
    Expands 52 blendshape features into temporal features used during training:
      52 current + 52 delta + 52 accel + 5 extra = 161
    The final 165-d vector appends the 4 geometric ratios.
    """

    def __init__(self, window_size: int = 5):
        self.window_size = window_size
        self.history = deque(maxlen=window_size)
        self.num_blendshapes = 52

    def extract(self, current_blendshapes: np.ndarray) -> np.ndarray:
        if current_blendshapes.shape[0] != self.num_blendshapes:
            raise ValueError(
                f"Expected {self.num_blendshapes} blendshapes, got {current_blendshapes.shape[0]}"
            )

        self.history.append(current_blendshapes)

        current = current_blendshapes
        delta = np.zeros(self.num_blendshapes, dtype=np.float32)
        accel = np.zeros(self.num_blendshapes, dtype=np.float32)

        if len(self.history) >= 2:
            prev = self.history[-2]
            delta = current - prev

        if len(self.history) >= 3:
            prev_prev = self.history[-3]
            accel = current - 2 * prev + prev_prev

        extra_features = np.zeros(5, dtype=np.float32)
        return np.concatenate([current, delta, accel, extra_features])

def _vec3(lm) -> np.ndarray:
    """Convert a MediaPipe landmark to a (3,) float64 array."""
    return np.array([lm.x, lm.y, lm.z], dtype=np.float64)


def _dist3(landmarks, i: int, j: int) -> float:
    """3D Euclidean distance between two landmark indices."""
    return float(np.linalg.norm(_vec3(landmarks[i]) - _vec3(landmarks[j])))


def calculate_ratios(landmarks) -> list[float]:
    """
    Compute 4 IOD-normalised geometric ratios — identical to training Block 1.

    EAR  : Eye Aspect Ratio (averaged left+right)
    MAR  : Mouth Aspect Ratio
    brow_to_eye_dist : mean brow-to-eye distance / IOD  ← fixes "face lift" bug
    mouth_pull       : mouth width / IOD

    All distances use 3D Euclidean to handle depth during head tilts.
    """
    iod = _dist3(landmarks, INNER_LEFT_EYE_IDX, INNER_RIGHT_EYE_IDX) + 1e-6

    # EAR: vertical eye height / horizontal eye width (both eyes averaged)
    ear = (
        (_dist3(landmarks, 159, 145) / (_dist3(landmarks, 133, 33) + 1e-6)) +
        (_dist3(landmarks, 386, 374) / (_dist3(landmarks, 362, 263) + 1e-6))
    ) / 2.0

    # MAR: lip gap / lip width
    mar = _dist3(landmarks, 13, 14) / (_dist3(landmarks, 61, 291) + 1e-6)

    # Brow-to-eye (normalised by IOD — this is the "face lift" fix)
    brow_to_eye = (
        (_dist3(landmarks, 107, 33) + _dist3(landmarks, 336, 362)) / 2.0
    ) / iod

    # Mouth width / IOD
    mouth_pull = _dist3(landmarks, 61, 291) / iod

    return [ear, mar, brow_to_eye, mouth_pull]


def verify_blendshape_order(result) -> list[str]:
    """
    Returns the actual blendshape names returned by MediaPipe in their
    detection order. Compare with CANONICAL_BLENDSHAPE_NAMES to catch drift.
    """
    if not result.face_blendshapes:
        return []
    return [bs.category_name for bs in result.face_blendshapes[0]]


def build_landmarker(model_path: str) -> Any:
    """Constructs a FaceLandmarker in IMAGE mode (Task API, NOT legacy FaceMesh)."""
    base_options = mp_python.BaseOptions(model_asset_path=str(model_path))
    options = mp_vision.FaceLandmarkerOptions(
        base_options=base_options,
        output_face_blendshapes=True,
        output_facial_transformation_matrixes=True,   # need for pitch check
        num_faces=1,
        min_face_detection_confidence=0.35,
        min_face_presence_confidence=0.35,
        min_tracking_confidence=0.35,
        running_mode=mp.tasks.vision.RunningMode.IMAGE,
    )
    return mp_vision.FaceLandmarker.create_from_options(options)


def extract_pitch_from_matrix(transform_matrix) -> float | None:
    """
    Extract pitch angle (degrees) from the 4×4 facial transformation matrix.
    Positive pitch = looking up (the "face lift" direction).
    """
    if transform_matrix is None or len(transform_matrix) == 0:
        return None
    # Matrix is a flat list of 16 floats (row-major 4×4)
    m = np.array(transform_matrix).reshape(4, 4)
    # Pitch from rotation matrix: arcsin(-m[2,1])  (ZYX Euler convention)
    try:
        pitch_rad = math.asin(float(np.clip(-m[2, 1], -1.0, 1.0)))
        return math.degrees(pitch_rad)
    except Exception:
        return None


def extract_features_from_image(
    image_path: Path,
    landmarker: Any,
    order_checked: list[bool],   # mutable flag — checked once per run
) -> tuple[np.ndarray | None, float | None]:
    """
    Returns:
        features : (56,) float32 or None if no face detected
        pitch_deg: head pitch in degrees or None
    """
    bgr = cv2.imread(str(image_path))
    if bgr is None:
        return None, None

    def _run(img_bgr):
        rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = landmarker.detect(mp_img)

        if not (result.face_blendshapes and result.face_landmarks):
            return None, None

        # ── LEAKAGE GUARD: verify canonical order once ───────────────────────
        if not order_checked[0]:
            actual = verify_blendshape_order(result)
            if actual != CANONICAL_BLENDSHAPE_NAMES:
                mismatches = [
                    (i, a, b)
                    for i, (a, b) in enumerate(zip(actual, CANONICAL_BLENDSHAPE_NAMES))
                    if a != b
                ]
                if mismatches:
                    print("\n[CRITICAL] Blendshape order mismatch — feature drift detected!")
                    for i, actual_name, expected_name in mismatches[:5]:
                        print(f"  idx={i}: got '{actual_name}', expected '{expected_name}'")
                    sys.exit(1)
                else:
                    print("[OK] Blendshape index order matches CANONICAL_BLENDSHAPE_NAMES")
            order_checked[0] = True

        scores = [bs.score for bs in result.face_blendshapes[0]]   # 52 values
        ratios = calculate_ratios(result.face_landmarks[0])         # 4 values
        features = np.array(scores + ratios, dtype=np.float32)      # (56,)

        pitch = None
        if result.facial_transformation_matrixes:
            pitch = extract_pitch_from_matrix(
                result.facial_transformation_matrixes[0].data
            )

        return features, pitch

    # Attempt 1: raw
    feat, pitch = _run(bgr)
    if feat is not None:
        return feat, pitch

    # Attempt 2: CLAHE
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    feat, pitch = _run(cv2.cvtColor(lab, cv2.COLOR_LAB2BGR))
    if feat is not None:
        return feat, pitch

    # Attempt 3: centre-crop 90%
    h, w = bgr.shape[:2]
    m = 0.05
    feat, pitch = _run(bgr[int(h*m):int(h*(1-m)), int(w*m):int(w*(1-m))])
    return feat, pitch


# ─────────────────────────────────────────────────────────────────────────────
# 2.  TEMPORAL SMOOTHER  (EMA — identical to training Block 5)
# ─────────────────────────────────────────────────────────────────────────────

class TemporalSmoother:
    """
    EMA smoother for per-frame probability vectors.
    Used here to simulate smoothed accuracy on RAF-DB image sequences
    (images in the same emotion folder are treated as a pseudo-sequence).
    """

    def __init__(
        self,
        window: int = 7,
        strategy: str = "ema",
        ema_alpha: float = 0.35,
        num_classes: int = NUM_CLASSES,
        emotion_names: list[str] | None = None,
    ):
        assert strategy in ("sma", "ema", "majority_vote")
        self.window = window
        self.strategy = strategy
        self.alpha = ema_alpha
        self.num_classes = num_classes
        self.names = emotion_names or [str(i) for i in range(num_classes)]
        self._prob_buf = deque(maxlen=window)
        self._vote_buf = deque(maxlen=window)
        self._ema = np.ones(num_classes, dtype=np.float32) / num_classes
        self.smoothed = np.ones(num_classes, dtype=np.float32) / num_classes
        self.current_label = -1
        self.confidence = 0.0

    def update(self, raw_probs: np.ndarray) -> np.ndarray:
        probs = np.asarray(raw_probs, dtype=np.float32).flatten()
        probs = probs / (probs.sum() + 1e-9)
        self._prob_buf.append(probs)
        self._vote_buf.append(int(np.argmax(probs)))

        if self.strategy == "sma":
            self.smoothed = np.mean(list(self._prob_buf), axis=0)
        elif self.strategy == "ema":
            self._ema = self.alpha * probs + (1 - self.alpha) * self._ema
            self.smoothed = self._ema / (self._ema.sum() + 1e-9)
        elif self.strategy == "majority_vote":
            votes = list(self._vote_buf)
            winner = max(set(votes), key=votes.count)
            avg = np.mean(list(self._prob_buf), axis=0)
            mask = np.zeros(self.num_classes, dtype=np.float32)
            mask[winner] = avg[winner]
            rem = (1 - mask[winner]) / (self.num_classes - 1 + 1e-9)
            for i in range(self.num_classes):
                if i != winner:
                    mask[i] = rem
            self.smoothed = mask

        self.current_label = int(np.argmax(self.smoothed))
        self.confidence = float(self.smoothed[self.current_label])
        return self.smoothed

    def reset(self):
        self._prob_buf.clear()
        self._vote_buf.clear()
        self._ema = np.ones(self.num_classes, dtype=np.float32) / self.num_classes


# ─────────────────────────────────────────────────────────────────────────────
# 3.  INT8 TFLITE INFERENCE
# ─────────────────────────────────────────────────────────────────────────────

class INT8TFLiteInferenceEngine:
    """
    Loads the INT8 TFLite model and the blendshape_metadata.json scaler.
    Implements full manual quantise → invoke → dequantise pipeline.
    """

    def __init__(self, tflite_path: str, metadata_path: str):
        # Load TFLite interpreter
        self.interp = tf.lite.Interpreter(model_path=str(tflite_path))
        self.interp.allocate_tensors()
        self._inp = self.interp.get_input_details()[0]
        self._out = self.interp.get_output_details()[0]

        # Quantisation params from model tensors
        self.inp_scale, self.inp_zero = self._inp["quantization"]
        self.out_scale, self.out_zero = self._out["quantization"]

        # Scaler params from metadata (must match training scaler exactly)
        with open(metadata_path) as f:
            meta = json.load(f)
        self._mean  = np.array(meta["mean"],  dtype=np.float32)
        self._scale = np.array(meta["scale"], dtype=np.float32)

        if len(self._mean) != INPUT_DIM or len(self._scale) != INPUT_DIM:
            raise ValueError(
                "Scaler metadata dimension mismatch. "
                f"Expected {INPUT_DIM}, got mean={len(self._mean)} scale={len(self._scale)}"
            )

        # Validate feature order in metadata against canonical list
        if "feature_names" in meta:
            meta_feats = meta["feature_names"]
            if meta_feats != FEATURE_NAMES:
                mismatches = [
                    (i, a, b)
                    for i, (a, b) in enumerate(zip(meta_feats, FEATURE_NAMES))
                    if a != b
                ]
                if mismatches:
                    print("[WARN] metadata feature_names differ from CANONICAL:")
                    for i, a, b in mismatches[:5]:
                        print(f"  idx={i}: metadata='{a}', canonical='{b}'")

        # Quantisation params from metadata (for cross-validation)
        meta_inp_q = meta.get("inp_quantization", {})
        meta_out_q = meta.get("out_quantization", {})
        if meta_inp_q:
            assert abs(meta_inp_q["scale"] - self.inp_scale) < 1e-6, (
                f"inp_scale mismatch: model={self.inp_scale}, meta={meta_inp_q['scale']}"
            )
        print(f"[OK] INT8 model loaded. "
              f"inp_scale={self.inp_scale:.6f}, out_scale={self.out_scale:.6f}")
        print(f"     Input  dtype={self._inp['dtype'].__name__}, "
              f"shape={self._inp['shape']}")
        print(f"     Output dtype={self._out['dtype'].__name__}, "
              f"shape={self._out['shape']}")
        print(f"     Scaler: mean[0]={self._mean[0]:.4f}, "
              f"scale[0]={self._scale[0]:.4f}")

    def normalise(self, features: np.ndarray) -> np.ndarray:
        """Apply StandardScaler: z = (x - mean) / scale."""
        return (features - self._mean) / self._scale

    def quantise(self, x_norm: np.ndarray) -> np.ndarray:
        """float32 → int8 using model's stored scale and zero_point."""
        return np.clip(
            np.round(x_norm / self.inp_scale + self.inp_zero), -128, 127
        ).astype(np.int8)

    def dequantise_output(self, y_int8: np.ndarray) -> np.ndarray:
        """int8 → float32 probabilities using output quantisation params."""
        return (y_int8.astype(np.float32) - self.out_zero) * self.out_scale

    def predict_proba(self, features: np.ndarray) -> np.ndarray:
        """
        Full pipeline: raw (165,) float32 → (5,) float32 probabilities.
        Thread-unsafe — do not share across threads.
        """
        x_norm = self.normalise(features)
        x_q    = self.quantise(x_norm).reshape(1, -1)
        self.interp.set_tensor(self._inp["index"], x_q)
        self.interp.invoke()
        y_q  = self.interp.get_tensor(self._out["index"])   # (1, 5) int8
        probs = self.dequantise_output(y_q[0])
        probs = np.clip(probs, 0.0, 1.0)
        # Re-normalise (INT8 rounding can shift sum off 1.0)
        total = probs.sum()
        return probs / total if total > 1e-6 else probs

    def predict(self, features: np.ndarray) -> int:
        return int(np.argmax(self.predict_proba(features)))


# ─────────────────────────────────────────────────────────────────────────────
# 4.  HEAD-POSE STABILITY (pitch regression test)
# ─────────────────────────────────────────────────────────────────────────────

def head_pose_stability_report(
    pitch_vals: list[float],
    predictions: list[int],
    true_labels: list[int],
    output_dir: Path,
):
    """
    Bin samples by pitch angle and report accuracy + "Unconfident" trigger rate.

    The "face lift" bug manifests as:
        • High pitch (looking up) → increased Anger/Fear/Sad predictions
        → elevated "Unconfident" rate for neutral faces

    Expectation after fix:
        • brow_to_eye_dist and mouth_pull are IOD-normalised, so pitch should
          not shift the feature distribution.
        • Accuracy and Unconfident-trigger-rate should be flat across bins.
    """
    bins     = [(-90, -20), (-20, -10), (-10, 10), (10, 20), (20, 90)]
    bin_lbls = ["↓ extreme", "↓ mild", "level", "↑ mild", "↑ extreme (face lift)"]

    rows = []
    for (lo, hi), lbl in zip(bins, bin_lbls):
        idxs = [
            i for i, p in enumerate(pitch_vals)
            if p is not None and lo <= p < hi
        ]
        if not idxs:
            continue
        preds = [predictions[i] for i in idxs]
        trues = [true_labels[i] for i in idxs]
        acc = sum(p == t for p, t in zip(preds, trues)) / len(idxs)
        # "Unconfident" trigger on neutral faces (true label == 3)
        neutral_idxs = [i for i, t in zip(idxs, trues) if t == 3]
        if neutral_idxs:
            n_preds = [predictions[i] for i in neutral_idxs]
            unconf_rate = sum(HR_MAP[p] == 0 for p in n_preds) / len(neutral_idxs)
        else:
            unconf_rate = float("nan")
        rows.append({
            "pitch_bin": lbl,
            "n_samples": len(idxs),
            "accuracy": round(acc, 4),
            "neutral_unconfident_rate": round(unconf_rate, 4) if not math.isnan(unconf_rate) else "n/a",
        })

    df = pd.DataFrame(rows)
    print("\n" + "=" * 70)
    print("HEAD-POSE STABILITY REPORT  (pitch angle bins)")
    print("=" * 70)
    print(df.to_string(index=False))
    print()
    print("  [PASS] if accuracy is flat (±5%) and unconfident_rate ≤ 0.15 across bins.")
    print("  [FAIL] if ↑ extreme bin shows ↑ unconfident_rate → 'face lift' bug present.")
    df.to_csv(output_dir / "head_pose_stability.csv", index=False)
    return df


# ─────────────────────────────────────────────────────────────────────────────
# 5.  QUANTISATION DELTA CHECK
# ─────────────────────────────────────────────────────────────────────────────

def quantisation_delta_check(
    engine: INT8TFLiteInferenceEngine,
    fp32_model,            # tf.keras.Model or None
    X_raw: np.ndarray,     # raw float32 features (165-d)
    y_true: np.ndarray,
) -> dict:
    """
    Measures FP32 vs INT8 accuracy delta.
    The delta should be < 1% for a well-calibrated INT8 model.
    If 60% accuracy is present in FP32, quantisation is NOT the culprit.
    """
    results = {}

    # INT8 predictions
    int8_preds = np.array([int(np.argmax(engine.predict_proba(x))) for x in X_raw])
    int8_acc   = float((int8_preds == y_true).mean())
    results["int8_accuracy"] = round(int8_acc, 4)

    if fp32_model is not None:
        # Apply same StandardScaler normalisation as INT8 path
        X_norm = engine.normalise(X_raw)
        fp32_probs = fp32_model.predict(X_norm, batch_size=512, verbose=0)
        fp32_preds = np.argmax(fp32_probs, axis=1)
        fp32_acc   = float((fp32_preds == y_true).mean())
        results["fp32_accuracy"] = round(fp32_acc, 4)
        results["delta_pct"]     = round(abs(fp32_acc - int8_acc) * 100, 3)
        print(f"\nFP32 accuracy   : {fp32_acc:.4f}")
        print(f"INT8 accuracy   : {int8_acc:.4f}")
        print(f"Delta           : {results['delta_pct']:.3f}%  (target: < 1%)")
        if results["delta_pct"] < 1.0:
            print("[PASS] Quantisation noise is within acceptable range.")
        else:
            print("[FAIL] Quantisation degrades accuracy > 1%. "
                  "Consider per-channel quantisation or FP16 fallback.")
    else:
        print(f"\nINT8 accuracy   : {int8_acc:.4f}  (no FP32 model provided for delta)")

    return results


# ─────────────────────────────────────────────────────────────────────────────
# 6.  CONFUSION-MATRIX + CLASSIFICATION REPORT PLOTTING
# ─────────────────────────────────────────────────────────────────────────────

def _try_import_plot_libs():
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        import seaborn as sns
        return plt, sns
    except ImportError:
        return None, None


def save_confusion_matrix(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    labels: list[str],
    title: str,
    out_path: Path,
):
    plt, sns = _try_import_plot_libs()
    cm = confusion_matrix(y_true, y_pred, normalize="true")
    cm_df = pd.DataFrame(cm.round(3), index=labels, columns=labels)
    print(f"\n{title}")
    print(cm_df.to_string())
    cm_df.to_csv(out_path.with_suffix(".csv"))

    if plt and sns:
        fig, ax = plt.subplots(figsize=(max(6, len(labels) * 1.5), max(5, len(labels) * 1.3)))
        sns.heatmap(
            cm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=labels, yticklabels=labels, ax=ax,
            linewidths=0.5, cbar_kws={"label": "proportion"},
        )
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title(title)
        plt.tight_layout()
        fig.savefig(out_path, dpi=150)
        plt.close(fig)
        print(f"  → Saved: {out_path}")
    return cm


# ─────────────────────────────────────────────────────────────────────────────
# 7.  TEMPORAL SMOOTHING SIMULATION ON RAF-DB IMAGE SEQUENCES
# ─────────────────────────────────────────────────────────────────────────────

def simulate_temporal_smoothing(
    probs_by_class: dict[int, list[np.ndarray]],
    strategies: list[str] = ("ema",),
    window: int = 7,
    ema_alpha: float = 0.35,
) -> dict:
    """
    Treat all images from the same emotion class as a pseudo-sequence
    (RAF-DB images are independent, but we can still measure whether EMA
    filtering improves or degrades per-class accuracy).

    Returns per-strategy accuracy dict.
    """
    results = {}
    for strategy in strategies:
        correct = 0
        total   = 0
        for true_label, prob_seq in probs_by_class.items():
            smoother = TemporalSmoother(
                window=window,
                strategy=strategy,
                ema_alpha=ema_alpha,
                num_classes=NUM_CLASSES,
                emotion_names=EMOTION_NAMES,
            )
            smoother.reset()
            for probs in prob_seq:
                smoother.update(probs)
                if smoother.current_label == true_label:
                    correct += 1
                total += 1
        results[strategy] = round(correct / max(total, 1), 4)
    return results


# ─────────────────────────────────────────────────────────────────────────────
# 8.  MAIN EVALUATION LOOP
# ─────────────────────────────────────────────────────────────────────────────

def evaluate(args):
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 70)
    print("RAF-DB FER Blendshape Evaluation — INT8 TFLite Pipeline")
    print("=" * 70)

    # ── Load artefacts ────────────────────────────────────────────────────────
    engine = INT8TFLiteInferenceEngine(args.tflite, args.metadata)

    fp32_model = None
    if args.fp32_keras and Path(args.fp32_keras).exists():
        print(f"\nLoading FP32 Keras model from {args.fp32_keras} ...")
        try:
            # This checkpoint contains a Lambda layer from training.
            # For trusted local artifacts, disable safe-mode for deserialization.
            fp32_model = tf.keras.models.load_model(
                args.fp32_keras,
                compile=False,
                safe_mode=False,
            )
            print("[OK] FP32 model loaded.")
        except TypeError:
            # Backward compatibility for older keras/tf that don't accept safe_mode.
            fp32_model = tf.keras.models.load_model(args.fp32_keras, compile=False)
            print("[OK] FP32 model loaded (legacy loader).")
        except Exception as exc:
            print(f"[WARN] Could not load FP32 model: {exc}")
            print("[WARN] Continuing with INT8-only evaluation; quantisation delta will be skipped.")
            fp32_model = None

    print(f"\nBuilding MediaPipe FaceLandmarker from {args.landmarker} ...")
    landmarker = build_landmarker(args.landmarker)
    order_checked = [False]   # mutable container for first-run index guard


    # ── Walk RAF-DB split (no 'test' subfolder) ─────────────────────────────
    rafdb_root = Path(args.rafdb_root)
    if not rafdb_root.exists():
        sys.exit(f"[ERROR] RAF-DB directory not found: {rafdb_root}")

    all_features: list[np.ndarray]   = []
    all_true:     list[int]          = []
    all_pitch:    list[float | None] = []
    all_paths:    list[str]          = []
    skipped = 0
    probs_by_class: dict[int, list[np.ndarray]] = defaultdict(list)
    temporal_extractor = TemporalFeatureExtractor(window_size=5)

    for emotion_name, label_idx in EMOTION_LABELS.items():
        class_dir = rafdb_root / emotion_name
        if not class_dir.exists():
            print(f"[WARN] Missing: {class_dir}")
            continue
        img_paths = sorted([
            p for p in class_dir.iterdir()
            if p.suffix.lower() in IMG_EXTENSIONS
        ])
        for img_path in tqdm(img_paths, desc=f"{emotion_name:>10}", unit="img"):
            feat, pitch = extract_features_from_image(img_path, landmarker, order_checked)
            if feat is None:
                skipped += 1
                continue
            expanded = temporal_extractor.extract(feat[:52])
            final_vector = np.concatenate([expanded, feat[52:]])
            all_features.append(final_vector.astype(np.float32))
            all_true.append(label_idx)
            all_pitch.append(pitch)
            all_paths.append(str(img_path))

    landmarker.close()

    total_attempted = len(all_features) + skipped
    print(f"\n  Extracted : {len(all_features):,} / {total_attempted:,} images")
    print(f"  Skipped   : {skipped:,}  ({100 * skipped / max(total_attempted, 1):.1f}%)")

    if not all_features:
        sys.exit("[ERROR] No features extracted — check paths and MediaPipe model.")

    X = np.stack(all_features, axis=0).astype(np.float32)   # (N, 165)
    y = np.array(all_true, dtype=np.int32)                   # (N,)

    # ── Per-sample INT8 inference ─────────────────────────────────────────────
    print("\nRunning INT8 inference on all samples ...")
    t0 = time.perf_counter()
    all_probs:  list[np.ndarray] = []
    int8_preds: list[int]        = []

    for feat, true_label in tqdm(zip(X, y), total=len(X), desc="INT8 inference"):
        probs = engine.predict_proba(feat)
        all_probs.append(probs)
        int8_preds.append(int(np.argmax(probs)))
        probs_by_class[int(true_label)].append(probs)

    elapsed = time.perf_counter() - t0
    ms_per_frame = elapsed / len(X) * 1000
    print(f"  Inference time: {elapsed:.2f}s total, {ms_per_frame:.2f}ms/sample")

    int8_preds_arr = np.array(int8_preds, dtype=np.int32)

    # ── 5-Class Classification Report ────────────────────────────────────────
    print("\n" + "=" * 70)
    print("5-CLASS CLASSIFICATION REPORT  (Anger / Fear / Happy / Neutral / Sad)")
    print("=" * 70)
    report_5 = classification_report(
        y, int8_preds_arr, target_names=EMOTION_NAMES, digits=4
    )
    print(report_5)
    (output_dir / "classification_report_5class.txt").write_text(report_5)

    save_confusion_matrix(
        y, int8_preds_arr, EMOTION_NAMES,
        "Normalised Confusion Matrix — 5-Class",
        output_dir / "confusion_matrix_5class.png",
    )

    # ── 3-Class "Virtual HR" Report ───────────────────────────────────────────
    y_hr      = np.array([HR_MAP[v] for v in y],              dtype=np.int32)
    preds_hr  = np.array([HR_MAP[v] for v in int8_preds_arr], dtype=np.int32)

    print("\n" + "=" * 70)
    print("3-CLASS VIRTUAL HR REPORT  (Unconfident / Happy / Neutral)")
    print("=" * 70)
    print("  Class-mapping precision note:")
    print("    Anger+Fear+Sad → Unconfident (class 0)")
    print("    Happy          → Happy       (class 1)")
    print("    Neutral        → Neutral     (class 2)")
    print()
    report_3 = classification_report(
        y_hr, preds_hr, target_names=HR_NAMES, digits=4
    )
    print(report_3)
    (output_dir / "classification_report_3class.txt").write_text(report_3)

    save_confusion_matrix(
        y_hr, preds_hr, HR_NAMES,
        "Normalised Confusion Matrix — 3-Class Virtual HR",
        output_dir / "confusion_matrix_3class.png",
    )

    # ── Quantisation Delta ────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("QUANTISATION DELTA CHECK  (FP32 vs INT8)")
    print("=" * 70)
    delta_results = quantisation_delta_check(engine, fp32_model, X, y)
    (output_dir / "quantisation_delta.json").write_text(json.dumps(delta_results, indent=2))

    # ── Head-Pose Stability ───────────────────────────────────────────────────
    valid_pitch = [(p, pred, t) for p, pred, t in zip(all_pitch, int8_preds, all_true) if p is not None]
    if valid_pitch:
        pitches, hpreds, htrues = zip(*valid_pitch)
        head_pose_stability_report(
            list(pitches), list(hpreds), list(htrues), output_dir
        )
    else:
        print("\n[WARN] No pitch data available — "
              "ensure output_facial_transformation_matrixes=True in landmarker options.")

    # ── Temporal Smoothing Simulation ─────────────────────────────────────────
    print("\n" + "=" * 70)
    print("TEMPORAL SMOOTHING ACCURACY  (pseudo-sequence per class, EMA α=0.35)")
    print("=" * 70)
    smooth_results = simulate_temporal_smoothing(
        probs_by_class,
        strategies=["sma", "ema", "majority_vote"],
        window=7,
        ema_alpha=0.35,
    )
    raw_acc = float((int8_preds_arr == y).mean())
    print(f"  Raw (no smoothing) : {raw_acc:.4f}")
    for strat, acc in smooth_results.items():
        delta = acc - raw_acc
        sign  = "+" if delta >= 0 else ""
        print(f"  {strat:<18}: {acc:.4f}  ({sign}{delta*100:.2f}%)")
    (output_dir / "temporal_smoothing.json").write_text(
        json.dumps({"raw": raw_acc, **smooth_results}, indent=2)
    )

    # ── Class Mapping Precision Analysis ─────────────────────────────────────
    print("\n" + "=" * 70)
    print("CLASS MAPPING ANALYSIS — Does early merging hurt precision?")
    print("=" * 70)
    print()
    prec5, rec5, f5, supp5 = precision_recall_fscore_support(
        y, int8_preds_arr, labels=list(range(NUM_CLASSES)), zero_division=0
    )
    print("  5-class model granularity:")
    print(f"  {'Class':<12} {'Prec':>8} {'Recall':>8} {'F1':>8} {'Support':>10}")
    for i, name in enumerate(EMOTION_NAMES):
        print(f"  {name:<12} {prec5[i]:>8.4f} {rec5[i]:>8.4f} {f5[i]:>8.4f} {int(supp5[i]):>10}")

    prec3, rec3, f3, supp3 = precision_recall_fscore_support(
        y_hr, preds_hr, labels=list(range(len(HR_NAMES))), zero_division=0
    )
    print()
    print("  3-class Virtual HR after merge:")
    print(f"  {'Class':<12} {'Prec':>8} {'Recall':>8} {'F1':>8} {'Support':>10}")
    for i, name in enumerate(HR_NAMES):
        print(f"  {name:<12} {prec3[i]:>8.4f} {rec3[i]:>8.4f} {f3[i]:>8.4f} {int(supp3[i]):>10}")

    print()
    print("  Interpretation:")
    unconf_prec_5class = (prec5[0] + prec5[1] + prec5[4]) / 3
    unconf_prec_3class = prec3[0]
    print(f"    Avg 'Unconfident' precision (5-class): {unconf_prec_5class:.4f}")
    print(f"    'Unconfident' precision (3-class merge): {unconf_prec_3class:.4f}")
    if unconf_prec_3class > unconf_prec_5class:
        print("    → Merge IMPROVES Unconfident precision (distinct signals averaged up)")
    else:
        print("    → Merge LOSES precision — consider keeping 5-class output and "
              "mapping in post-processing only for display.")

    # ── Final Summary ─────────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("SUMMARY")
    print("=" * 70)
    overall_acc = float((int8_preds_arr == y).mean())
    print(f"  Overall 5-class accuracy   : {overall_acc:.4f}  ({overall_acc*100:.1f}%)")
    print(f"  INT8 model (delta vs FP32) : {delta_results.get('delta_pct', 'n/a')}%)")
    print(f"  Best temporal strategy     : {max(smooth_results, key=smooth_results.get)}"
          f"  ({max(smooth_results.values()):.4f})")

    print()
    print("  Diagnoses:")
    if overall_acc < 0.65:
        print("  ⚠️  Accuracy below 65% — likely causes:")
        print("      1. Class imbalance in RAF-DB (check class distribution above)")
        print("      2. Domain shift: AffectNet (posed) → RAF-DB (in-the-wild)")
        print("      3. Fear/Sad confusion (low intra-class FACS separability)")
        print("      4. IOD normalisation applied at test but not training? "
              "Check scaler mean/scale values match training script Block 3.")
    if delta_results.get("delta_pct", 0) > 1.0:
        print("  ⚠️  Quantisation delta > 1% — recalibrate with more diverse "
              "representative_dataset samples (use full val split, not first 500).")
    print()
    print(f"  All outputs saved to: {output_dir.resolve()}")


# ─────────────────────────────────────────────────────────────────────────────
# 9.  CLI
# ─────────────────────────────────────────────────────────────────────────────

def build_arg_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(
        description="Rigorous RAF-DB evaluation for the FER blendshape INT8 pipeline."
    )
    p.add_argument("--rafdb_root",  required=True,
                   help="Path to RAF-DB root (contains test/ subdirectory)")
    p.add_argument("--tflite",      required=True,
                   help="Path to fer_blendshape_int8.tflite")
    p.add_argument("--metadata",    required=True,
                   help="Path to blendshape_metadata.json")
    p.add_argument("--landmarker",  required=True,
                   help="Path to face_landmarker.task (MediaPipe Task API model)")
    p.add_argument("--fp32_keras",  default=None,
                   help="(Optional) Path to best_fer_model.keras for quant delta check")
    p.add_argument("--output_dir",  default="./eval_results",
                   help="Directory to write reports, CSVs, and confusion-matrix PNGs")
    return p


if __name__ == "__main__":
    parser = build_arg_parser()
    # Provide default arguments when running in Colab
    args = parser.parse_args([
        "--rafdb_root", "./testing rafdb",
        "--tflite", "./fer_blendshape_int8.tflite",
        "--metadata", "./blendshape_metadata.json",
        "--landmarker", "face_landmarker.task",
        "--fp32_keras", "./best_fer_model.keras", # Optional, add if file exists
        "--output_dir", "./eval_results"
    ])
    evaluate(args)



RAF-DB FER Blendshape Evaluation — INT8 TFLite Pipeline
[OK] INT8 model loaded. inp_scale=0.245171, out_scale=0.035079
     Input  dtype=int8, shape=[  1 165]
     Output dtype=int8, shape=[1 5]
     Scaler: mean[0]=0.0000, scale[0]=0.0000

Loading FP32 Keras model from ./best_fer_model.keras ...
[OK] FP32 model loaded.

Building MediaPipe FaceLandmarker from face_landmarker.task ...


       Sad: 100%|██████████| 355/355 [00:06<00:00, 51.80img/s]



  Extracted : 1,640 / 1,775 images
  Skipped   : 135  (7.6%)

Running INT8 inference on all samples ...


INT8 inference: 100%|██████████| 1640/1640 [00:00<00:00, 1945.61it/s]


  Inference time: 0.85s total, 0.52ms/sample

5-CLASS CLASSIFICATION REPORT  (Anger / Fear / Happy / Neutral / Sad)
              precision    recall  f1-score   support

       Anger     0.5386    0.9128    0.6775       298
        Fear     0.6849    0.6455    0.6646       330
       Happy     0.8653    0.7428    0.7994       346
     Neutral     0.6152    0.8979    0.7302       333
         Sad     0.7073    0.0871    0.1551       333

    accuracy                         0.6524      1640
   macro avg     0.6823    0.6572    0.6053      1640
weighted avg     0.6868    0.6524    0.6052      1640


Normalised Confusion Matrix — 5-Class
         Anger   Fear  Happy  Neutral    Sad
Anger    0.913  0.034  0.037    0.010  0.007
Fear     0.279  0.645  0.048    0.027  0.000
Happy    0.046  0.162  0.743    0.026  0.023
Neutral  0.081  0.015  0.000    0.898  0.006
Sad      0.294  0.081  0.039    0.498  0.087
  → Saved: eval_results/confusion_matrix_5class.png

3-CLASS VIRTUAL HR REPORT  (Uncon